In [1]:
# En esta versión, cambio de FFNN a LSTM
# Se predicen los RENDIMIENTOS

# Librerías y Módulos

In [2]:
import sys

# Obtener la lista de kernels
kernels = !jupyter kernelspec list

# Mostrar la lista de kernels para referencia
print(kernels)

# Buscar si 'mi_entorno' está en la lista de kernels
kernel_name = "mi_entorno"
kernel_found = False

for line in kernels:
    if kernel_name in line:
        kernel_found = True
        break

if not kernel_found:
    print(f'Kernel debe ser: {kernel_name}')
    #sys.exit()

print(f'Kernel correcto: {kernel_name}')


['"jupyter" no se reconoce como un comando interno o externo,', 'programa o archivo por lotes ejecutable.']
Kernel debe ser: mi_entorno
Kernel correcto: mi_entorno


In [3]:
#sys.exit('predecir rendimietos y no precios')

In [4]:
sys.path.insert(0, '')
from Clase_Valor_240902 import Valor
from Transversal import leer_activos, leer_parametros, leer_configuracion

## Librerías

In [5]:
import pandas as pd
import numpy as np
import datetime as dt
import itertools
import os
import matplotlib.pyplot as plt
import yfinance as yf

import pickle
import tqdm
import time

from sklearn.model_selection import train_test_split

#import import_ipynb # permite importar módulos ipynb
import warnings
warnings.filterwarnings("ignore")

import mip

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

from sklearn import linear_model

import ipywidgets as widgets
from IPython.display import display

# Parámetros

In [6]:
carpeta_input, cofre, seguimiento, saving_step = leer_parametros()
output_level = leer_configuracion()

In [7]:
# Parámetros para construir matrices

cofre = '../Cofre/Red_Neuronal/'
print('Ejemplo')
lista_campos = ['Fibonacci', 'Media_Movil_300_Close', ''] # ejemplo...todos los campos que contengan: PARAMETRO (Filtrar campos)... '' incluye todos los campos
excluir_campos = ['Media_Movil_2_', 'Media_Movil_1_']

Ejemplo


Ejecución

In [8]:
reiniciar_clusters = False # True para dejar las iteraciones de clusters en 0 para los activos seleccionados

# Lectura de activos

In [9]:
df_activos = leer_activos(carpeta_input)
df_activos

,SIMBOLO,NOMBRE
0,NFLX,"Netflix, Inc."
1,AAPL,Apple Inc.
2,WMT,Walmart
3,TSLA,Tesla
4,AMZN,Amazon
5,BRK-A,Berkshire Hathaway


# Funciones

In [10]:
def pickle_act(file_name, variable = None, mode = 'open', eliminar_si_problemas = False):
    
    """
    Guarda o carga una variable utilizando la biblioteca pickle.

    Parameters:
        - file_path (str): La ruta al archivo pickle.
        - variable: La variable a guardar (si mode='save') o None (si mode='open').
        - mode (str): 'save' para guardar la variable, 'open' para cargar la variable.

    Returns:
        La variable cargada si mode='open' o None si mode='save'.
    """
    
    dic_mode = {'save': 'wb', 'open': 'rb'}
    
    #while True:
    #    try:
    with open(f'{file_name}.pkl', dic_mode[mode]) as file:
        if mode == 'save':
            pickle.dump(variable, file)
            return None
        else:
            #print('file en funciones transversales', f'{file_name}.pkl')
            if eliminar_si_problemas:
                try:
                    variable = pickle.load(file)
                except:
                    os.remove(f'{file_name}.pkl')
                    return pd.DataFrame()
            else:
                variable = pickle.load(file)
            return variable

# Nueva versión (LSTM y RLM dicotómica para búsqueda de hiperparámetros)

Ejemplo básico aleatorio

In [11]:
"""
# Generar datos de ejemplo: secuencia de números
data = np.array([i for i in range(1000)], dtype=float)
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, 1))

# Definir el tamaño de las secuencias (en este caso simplemente se crean matrices aleatorias)
timesteps = 10
X = []
y = []

for i in range(len(data) - timesteps):
    X.append(data[i:i + timesteps])
    y.append(data[i + timesteps])

X = np.array(X)
y = np.array(y)

# Redimensionar X para que tenga forma (muestras, timesteps, características)
X = np.reshape(X, (X.shape[0], timesteps))

ejecutar_ejemplo = False
if ejecutar_ejemplo:
    # Crear el modelo LSTM
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=False, input_shape=(timesteps, 1)))
    model.add(Dense(units=1))

    # Compilar el modelo
    model.compile(optimizer='adam', loss='mean_squared_error')

    # Entrenar el modelo
    model.fit(X, y, epochs = 10, batch_size = 32)

    # Predecir utilizando el modelo entrenado
    predicciones = model.predict(X)

    # Invertir la normalización para ver los valores originales
    predicciones_originales = scaler.inverse_transform(predicciones)

    print(predicciones_originales[:5])  # Imprimir primeras 5 predicciones
"""

"\n# Generar datos de ejemplo: secuencia de números\ndata = np.array([i for i in range(1000)], dtype=float)\nscaler = MinMaxScaler(feature_range=(0, 1))\ndata = scaler.fit_transform(data.reshape(-1, 1))\n\n# Definir el tamaño de las secuencias (en este caso simplemente se crean matrices aleatorias)\ntimesteps = 10\nX = []\ny = []\n\nfor i in range(len(data) - timesteps):\n    X.append(data[i:i + timesteps])\n    y.append(data[i + timesteps])\n\nX = np.array(X)\ny = np.array(y)\n\n# Redimensionar X para que tenga forma (muestras, timesteps, características)\nX = np.reshape(X, (X.shape[0], timesteps))\n\nejecutar_ejemplo = False\nif ejecutar_ejemplo:\n    # Crear el modelo LSTM\n    model = Sequential()\n    model.add(LSTM(units=50, return_sequences=False, input_shape=(timesteps, 1)))\n    model.add(Dense(units=1))\n\n    # Compilar el modelo\n    model.compile(optimizer='adam', loss='mean_squared_error')\n\n    # Entrenar el modelo\n    model.fit(X, y, epochs = 10, batch_size = 32)\n\n 

# Construir matrices (input y output)

In [12]:
df_activos

,SIMBOLO,NOMBRE
0,NFLX,"Netflix, Inc."
1,AAPL,Apple Inc.
2,WMT,Walmart
3,TSLA,Tesla
4,AMZN,Amazon
5,BRK-A,Berkshire Hathaway


In [13]:
# Revisar: método construir_matrices(self)

# crea funcion tal que, en el campo name del df, si cualquier campo de la lista_campos es un substring de NAME, entonces se filtra
def filtrar_campos(df, lista_campos, excluir_campos, eliminar_extras = True):
    df['FILTRO'] = False
    for campo in lista_campos:
        df['FILTRO'] = df['FILTRO'] | df['NAME'].str.contains(campo)
    
    for campo in excluir_campos:
        df['FILTRO'] = df['FILTRO'] & ~df['NAME'].str.contains(campo)
    
    for c in ['VALOR', 'DATE', 'Y']:
        if not eliminar_extras:
            break
        df['FILTRO'] = np.where(df['NAME'] == c, False, df['FILTRO'])
        
    return df[df['FILTRO']].drop(columns = 'FILTRO').reset_index(drop = True)

while True:

    # Matrices de input y de output
    cofre0 = '/'.join(cofre.split('/')[:-2]) + '/'
    df_raw_x, df_raw_y = pd.DataFrame(), pd.DataFrame()
    for i in range(len(df_activos)):
        simbolo, nombre = df_activos.loc[i]

        valor = Valor(simbolo, nombre, cofre0) # Si no existe el objeto, se crea 
        if len(valor.raw_x) == 0: # No hay datos que aportar
            continue

        new_raw_x = valor.raw_x.copy()
        new_raw_x['VALOR'] = simbolo
        
        print('SIMBOLO', simbolo, new_raw_x['DATE'].min(), new_raw_x['DATE'].max())
        
        new_raw_y = valor.adj_data[['Date', 'Close']].rename(columns = {'Date': 'DATE', 'Close': 'Y'})
        new_raw_y['VALOR'] = simbolo
        new_raw_y['Y'] = new_raw_y['Y'].pct_change().fillna(0) # Se considera el rendimiento como Y, no el precio
        
        df_raw_x = pd.concat([df_raw_x, new_raw_x], axis = 0)
        
        df_raw_y = pd.concat([df_raw_y, new_raw_y], axis = 0)
        #sys.exit()
        
        #display(df_raw_x)
        #display(new_raw_y)
        #sys.exit()

    df_raw_x = df_raw_x[['VALOR', 'DATE', 'NAME', 'X']]
    df_raw_x = filtrar_campos(df_raw_x, lista_campos, excluir_campos) # Filtro con lista_campos

    df_raw_x = df_raw_x.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()

    df = df_raw_x.merge(df_raw_y, on = ['VALOR', 'DATE'], how = 'outer')

    aprobado = True
    for c in list(set(df.columns) - {'DATE', 'VALOR'}):
        #print(c, df[c].min(), df[c].max())
        # Normalizacion
        
        if c != 'Y':
            #print(f'NORMALIZACION {c}!!!!')
            df[c] = (df[c] - df[c].min()) / (df[c].max() - df[c].min())
        
        if len(df[df[c].isna()]) > 0:
            print(f'El campo {c} tiene valores nulos')
            display(df[df[c].isna()][['DATE', 'VALOR', c]])
            if c == 'Y':
                df = df[df[c].notna()].reset_index(drop = True)
            else:
                aprobado = False
                sys.exit('Salida por campos nulos')
    if aprobado:
        break
    print('B. Espera de completitud de valores en Valor.py: Espera de 30s.')
    time.sleep(30)

df

../Cofre/Valor/NFLX
<_io.BufferedReader name='../Cofre/Valor/NFLX.pkl'>
SIMBOLO NFLX 2002-05-23 00:00:00 2025-04-16 00:00:00
../Cofre/Valor/AAPL
<_io.BufferedReader name='../Cofre/Valor/AAPL.pkl'>
SIMBOLO AAPL 1980-12-12 00:00:00 2025-04-16 00:00:00
../Cofre/Valor/WMT
<_io.BufferedReader name='../Cofre/Valor/WMT.pkl'>
SIMBOLO WMT 1972-08-25 00:00:00 2025-04-16 00:00:00
../Cofre/Valor/TSLA
<_io.BufferedReader name='../Cofre/Valor/TSLA.pkl'>
SIMBOLO TSLA 2010-06-29 00:00:00 2025-04-16 00:00:00
../Cofre/Valor/AMZN
<_io.BufferedReader name='../Cofre/Valor/AMZN.pkl'>
SIMBOLO AMZN 1997-05-15 00:00:00 2025-04-16 00:00:00
../Cofre/Valor/BRK-A
<_io.BufferedReader name='../Cofre/Valor/BRK-A.pkl'>
SIMBOLO BRK-A 1980-03-17 00:00:00 2025-04-16 00:00:00
El campo Y tiene valores nulos


,DATE,VALOR,Y
11135,2025-02-17,AAPL,NaN
11155,2025-03-16,AAPL,NaN
11161,2025-03-23,AAPL,NaN
18163,2025-02-17,AMZN,NaN
29529,2025-02-17,BRK-A,NaN
35293,2025-02-17,NFLX,NaN
39018,2025-02-17,TSLA,NaN
52290,2025-02-17,WMT,NaN
52310,2025-03-16,WMT,NaN
52316,2025-03-23,WMT,NaN


,VALOR,DATE,Fibonacci_100_cuerpo,Fibonacci_100_sombra,Fibonacci_150_cuerpo,Fibonacci_150_sombra,Fibonacci_200_cuerpo,Fibonacci_200_sombra,Fibonacci_20_cuerpo,Fibonacci_20_sombra,...,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.4_Close,Suavizamiento_Exponencial_0.4_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.5_rendimiento,Suavizamiento_Exponencial_0.6_Close,Suavizamiento_Exponencial_0.6_rendimiento,Suavizamiento_Exponencial_0.7_Close,Suavizamiento_Exponencial_0.7_rendimiento,Y
0,AAPL,1980-12-12,0.865602,0.862540,0.813430,1.0,0.777783,0.767567,0.708600,0.833575,...,0.514913,0.000000,0.522313,0.000000,0.523108,0.000000,0.526280,0.000000,0.531116,0.000000
1,AAPL,1980-12-15,0.865602,0.862540,0.813430,1.0,0.777783,0.767567,0.708600,0.833575,...,0.514913,0.000383,0.522313,0.000383,0.523108,0.000382,0.526280,0.000382,0.531116,-0.052171
2,AAPL,1980-12-16,0.747445,0.747624,0.649115,1.0,0.597239,0.588695,0.664191,0.780898,...,0.336816,0.000375,0.384351,0.000373,0.411902,0.000370,0.433282,0.000368,0.451338,-0.073398
3,AAPL,1980-12-17,0.881423,0.878635,0.835432,1.0,0.801959,0.792620,0.714547,0.840953,...,0.315077,0.000360,0.361898,0.000354,0.389279,0.000349,0.410579,0.000345,0.428616,0.024751
4,AAPL,1980-12-18,0.900415,0.897955,0.861844,1.0,0.830979,0.822693,0.721685,0.849809,...,0.400375,0.000354,0.452244,0.000349,0.482572,0.000346,0.506472,0.000344,0.526860,0.028993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52320,WMT,2025-04-10,0.902695,0.902548,0.751705,1.0,0.727387,0.713876,0.744877,0.832835,...,0.581362,0.827054,0.598602,0.828674,0.606676,0.831994,0.615578,0.836566,0.624710,0.011272
52321,WMT,2025-04-11,0.578585,0.566887,0.363357,1.0,0.410312,0.376073,0.417610,0.507573,...,0.572972,0.844303,0.580010,0.848736,0.576905,0.853401,0.574056,0.857648,0.571261,0.024170
52322,WMT,2025-04-14,0.578585,0.566887,0.363357,1.0,0.410312,0.374874,0.462541,0.538529,...,0.580307,0.863065,0.582497,0.869267,0.575766,0.874547,0.571241,0.878636,0.569031,0.020797
52323,WMT,2025-04-15,0.914714,0.914995,0.765760,1.0,0.739145,0.726277,0.706854,0.829550,...,0.581988,0.881736,0.580423,0.888784,0.571602,0.894094,0.566508,0.897854,0.564753,-0.008023


Separación de sets

In [14]:
df = df.sort_values(['DATE']).reset_index(drop = True)
df

,VALOR,DATE,Fibonacci_100_cuerpo,Fibonacci_100_sombra,Fibonacci_150_cuerpo,Fibonacci_150_sombra,Fibonacci_200_cuerpo,Fibonacci_200_sombra,Fibonacci_20_cuerpo,Fibonacci_20_sombra,...,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.4_Close,Suavizamiento_Exponencial_0.4_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.5_rendimiento,Suavizamiento_Exponencial_0.6_Close,Suavizamiento_Exponencial_0.6_rendimiento,Suavizamiento_Exponencial_0.7_Close,Suavizamiento_Exponencial_0.7_rendimiento,Y
0,WMT,1972-08-25,0.865602,0.862540,0.813430,1.0,0.777783,0.767567,0.708600,0.833575,...,0.514913,0.000000,0.522313,0.000000,0.523108,0.000000,0.526280,0.000000,0.531116,0.000000
1,WMT,1972-08-28,0.865602,0.862540,0.813430,1.0,0.777783,0.767567,0.708600,0.833575,...,0.514913,0.000111,0.522313,0.000111,0.523108,0.000111,0.526280,0.000111,0.531116,-0.003770
2,WMT,1972-08-29,0.776994,0.769181,0.690207,1.0,0.642390,0.622250,0.675297,0.790780,...,0.502042,0.000111,0.512343,0.000111,0.515071,0.000111,0.519560,0.000111,0.525351,-0.011401
3,WMT,1972-08-30,0.776994,0.769181,0.690207,1.0,0.642390,0.622250,0.675297,0.790780,...,0.494227,0.000111,0.504271,0.000110,0.506938,0.000110,0.511398,0.000110,0.517183,0.000000
4,WMT,1972-08-31,0.688114,0.839164,0.566605,1.0,0.506581,0.731182,0.641892,0.822860,...,0.500433,0.000110,0.511488,0.000110,0.515023,0.000110,0.520328,0.000109,0.526936,-0.015407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52320,BRK-A,2025-04-16,0.370562,0.350725,0.125000,1.0,0.225551,0.180685,0.485035,0.569030,...,0.533502,0.981115,0.537219,0.982828,0.533646,0.983836,0.533041,0.984109,0.534916,-1.000000
52321,TSLA,2025-04-16,0.789406,0.789617,0.591314,1.0,0.557278,0.535972,0.597125,0.715420,...,0.531441,0.522137,0.535462,0.521940,0.532930,0.521736,0.534128,0.521463,0.538289,-1.000000
52322,NFLX,2025-04-16,0.549626,0.547599,0.475379,1.0,0.474340,0.443582,0.545880,0.635720,...,0.580189,0.893684,0.585725,0.898779,0.583326,0.903327,0.584044,0.907438,0.587088,-1.000000
52323,AAPL,2025-04-16,0.690042,0.688186,0.569286,1.0,0.509526,0.496178,0.598350,0.705205,...,0.547824,0.773296,0.551497,0.776105,0.544993,0.778360,0.541054,0.779633,0.540023,-1.000000


# 2. df_precios

In [15]:
df_precios = pd.DataFrame()
for i in range(len(df_activos)):
    simbolo, nombre = df_activos.loc[i]
    valor = Valor(simbolo, nombre, cofre0) # Si no existe el objeto, se crea 
    if len(valor.raw_x) == 0: # No hay datos que aportar
        continue
    df_adj_data = valor.adj_data
    df_adj_data = df_adj_data[['Date', 'Close']].rename(columns = {'Date': 'DATE', 'Close': 'PRECIO'})
    df_adj_data['VALOR'] = simbolo
    
    df_precios = pd.concat([df_precios, df_adj_data])

df_precios = df_precios.reset_index(drop = True)
df_precios['DATE'] = pd.to_datetime(df_precios['DATE'])
df_precios

../Cofre/Valor/NFLX
<_io.BufferedReader name='../Cofre/Valor/NFLX.pkl'>
../Cofre/Valor/AAPL
<_io.BufferedReader name='../Cofre/Valor/AAPL.pkl'>
../Cofre/Valor/WMT
<_io.BufferedReader name='../Cofre/Valor/WMT.pkl'>
../Cofre/Valor/TSLA
<_io.BufferedReader name='../Cofre/Valor/TSLA.pkl'>
../Cofre/Valor/AMZN
<_io.BufferedReader name='../Cofre/Valor/AMZN.pkl'>
../Cofre/Valor/BRK-A
<_io.BufferedReader name='../Cofre/Valor/BRK-A.pkl'>


,DATE,PRECIO,VALOR
0,2002-05-23,0.001124,NFLX
1,2002-05-24,0.001137,NFLX
2,2002-05-28,0.001087,NFLX
3,2002-05-29,0.001037,NFLX
4,2002-05-30,0.001007,NFLX
...,...,...,...
52320,2025-04-10,0.957885,BRK-A
52321,2025-04-11,0.970261,BRK-A
52322,2025-04-14,0.982374,BRK-A
52323,2025-04-15,0.980780,BRK-A


# 3. Train, Test & Control

In [16]:
df['VALOR'].unique()

array(['WMT', 'BRK-A', 'AAPL', 'AMZN', 'NFLX', 'TSLA'], dtype=object)

In [17]:
df_campos = pd.DataFrame({'NAME': list(df.columns)})
df_campos = filtrar_campos(df_campos, lista_campos, excluir_campos) # filtrar campos es la base de los campos ocupados...todos los modelos vendrán de un subconjunto de estos campos
set_inputs_raw = set(df_campos['NAME'].unique())
print('L', len(set_inputs_raw))
lista_inputs = list(set_inputs_raw) # Son todos los inputs disponibles (no los seleccionados)

print(len(lista_inputs))

df = df[['VALOR', 'DATE'] + lista_inputs + ['Y']]

L 58
58


In [18]:
train_size, test_size, control_size = 0.7, 0.25, 0.05 # Parámetros

In [19]:
version_control = ['desde_fecha', '2024-01-01'] # puede ser ['desde_fecha', fecha], ['particion', control_size]
# si es desde_fecha, los parámetros: train_size, test_size, control_size se ignoran

In [20]:
df_train, df_control, df_predict = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
for valor in df['VALOR'].unique():
    df_valor = df[df['VALOR'] == valor].reset_index(drop = True)
    df_valor = df_valor.sort_values(['DATE']).reset_index(drop = True)
    
    df_valor_predict = df_valor[df_valor.DATE == str(dt.datetime.now().date())].reset_index(drop = True) # df_predict inicial (primer día)
    df_predict = pd.concat([df_predict, df_valor_predict])
    
    df_valor = df_valor[df_valor.DATE < str(dt.datetime.now().date())] # df_valor sin el último día
    
    if version_control[0] == 'desde_fecha':
        df_train_valor, df_control_valor = df_valor[df_valor.DATE < version_control[1]], df_valor[df_valor.DATE >= version_control[1]]
    elif version_control[0] == 'particion':
        df_train_valor, df_control_valor = train_test_split(df_valor, test_size = version_control[1], shuffle = False) # shuffle = False -> control son los datos del final, no random
        
    print(valor, len(df_control_valor) / (len(df_valor)), df_train_valor['DATE'].min().date(), df_control_valor['DATE'].min().date(), df_control_valor['DATE'].max().date())
    df_train = pd.concat([df_train, df_train_valor])
    df_control = pd.concat([df_control, df_control_valor])

df_predict = df_predict.reset_index(drop = True)
display('Train', df_train.head(), 'Control', df_control.head())

WMT 0.02434061793519216 1972-08-25 2024-01-02 2025-04-15
BRK-A 0.028423090461105246 1980-03-17 2024-01-02 2025-04-15
AAPL 0.02890121689334288 1980-12-12 2024-01-02 2025-04-15
AMZN 0.045985193621867884 1997-05-15 2024-01-02 2025-04-15
NFLX 0.056056924678930926 2002-05-23 2024-01-02 2025-04-15
TSLA 0.0867579908675799 2010-06-29 2024-01-02 2025-04-15


'Train'

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
0,WMT,1972-08-25,0.750195,0.000000,0.0,0.490819,0.514913,0.000000,0.0,0.526280,...,0.372791,0.000000,0.000000,0.851575,0.0,0.847774,0.000000,0.000000,0.0,0.000000
1,WMT,1972-08-28,0.750195,0.000111,0.0,0.490819,0.514913,0.000112,0.0,0.526280,...,0.372791,0.000111,0.000111,0.851575,0.0,0.847774,0.000112,0.000117,0.0,-0.003770
2,WMT,1972-08-29,0.543212,0.000111,0.0,0.490819,0.502042,0.000112,0.0,0.519560,...,0.372791,0.000111,0.000111,0.759403,0.0,0.770814,0.000112,0.000117,0.0,-0.011401
3,WMT,1972-08-30,0.543212,0.000110,0.0,0.490819,0.494227,0.000112,0.0,0.511398,...,0.372791,0.000110,0.000110,0.759403,0.0,0.770814,0.000111,0.000117,0.0,0.000000
4,WMT,1972-08-31,0.698370,0.000109,0.0,0.490819,0.500433,0.000112,0.0,0.520328,...,0.372791,0.000110,0.000110,0.828496,0.0,0.828504,0.000111,0.000117,0.0,-0.015407


'Control'

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
12947,WMT,2024-01-02,0.689452,0.495204,0.543751,0.496080,0.522180,0.496549,0.525562,0.527858,...,0.382742,0.495909,0.495603,0.895627,0.497064,0.884556,0.496070,0.517045,0.504551,0.010403
12948,WMT,2024-01-03,0.688918,0.498897,0.544085,0.505876,0.530654,0.498481,0.525503,0.538038,...,0.385075,0.498805,0.498872,0.895673,0.498733,0.884594,0.498354,0.517498,0.504967,0.000063
12949,WMT,2024-01-04,0.675718,0.500026,0.544387,0.498214,0.525996,0.500032,0.525578,0.531050,...,0.386831,0.500269,0.500198,0.888715,0.500560,0.878784,0.499963,0.517930,0.505459,-0.009667
12950,WMT,2024-01-05,0.669298,0.496978,0.544640,0.491891,0.512771,0.500295,0.525402,0.517849,...,0.382952,0.498575,0.497822,0.883970,0.500484,0.874823,0.499628,0.518086,0.505789,-0.006656
12951,WMT,2024-01-08,0.678714,0.493755,0.544872,0.486011,0.506597,0.499838,0.525007,0.515789,...,0.380618,0.496075,0.494890,0.890928,0.499941,0.880632,0.498397,0.518061,0.505944,0.009827


In [21]:
df_predict

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
0,WMT,2025-04-16,0.315287,0.898531,0.841609,0.530392,0.553649,0.874106,0.936234,0.533791,...,0.424844,0.894900,0.897547,0.559680,0.888704,0.604052,0.882676,0.904394,0.871560,-1.0
1,BRK-A,2025-04-16,0.000000,0.984109,0.959531,0.505676,0.533502,0.981339,0.994095,0.533041,...,0.413792,0.982828,0.983836,0.346266,0.981260,0.299995,0.979589,1.000000,1.000000,-1.0
2,AAPL,2025-04-16,0.192950,0.779633,0.946700,0.505903,0.547824,0.779171,0.912739,0.541054,...,0.337233,0.776105,0.778360,0.679438,0.771807,0.686585,0.772116,0.872117,0.854461,-1.0
3,AMZN,2025-04-16,0.198028,0.747700,0.921331,0.441306,0.501394,0.765661,0.895558,0.507937,...,0.367655,0.752150,0.749969,0.668254,0.765854,0.694707,0.757304,0.853311,0.825805,-1.0
4,NFLX,2025-04-16,0.328496,0.907438,0.812975,0.518428,0.580189,0.887244,0.958466,0.584044,...,0.451696,0.898779,0.903327,0.540638,0.891011,0.588152,0.888707,0.929678,0.910260,-1.0
5,TSLA,2025-04-16,0.461068,0.521463,0.599324,0.436679,0.531441,0.526816,0.622115,0.534128,...,0.466136,0.521940,0.521736,0.779579,0.530979,0.756042,0.523038,0.594285,0.541519,-1.0


In [22]:
#sys.exit('Excelente para pasar a un GAN Clusterizado!! Ver abajo')

# 4. Clusters

In [23]:
df_activos

,SIMBOLO,NOMBRE
0,NFLX,"Netflix, Inc."
1,AAPL,Apple Inc.
2,WMT,Walmart
3,TSLA,Tesla
4,AMZN,Amazon
5,BRK-A,Berkshire Hathaway


In [24]:
delta_days = (dt.datetime.today() - dt.datetime(2025, 3, 20)).days
delta_days

27

In [25]:
n_clusters, n_iteraciones, max_iter = 10, int(1500 * delta_days), 500

In [26]:
df_date_max = df_train[['VALOR', 'DATE']].rename(columns = {'VALOR': 'SIMBOLO', 'DATE': 'DATE_MAX_TRAIN'})
df_date_max['DATE_MAX_TRAIN'] = pd.to_datetime(df_date_max['DATE_MAX_TRAIN'])
df_date_max = df_date_max[['SIMBOLO', 'DATE_MAX_TRAIN']].groupby('SIMBOLO').max().reset_index()

df_date_max

,SIMBOLO,DATE_MAX_TRAIN
0,AAPL,2023-12-29
1,AMZN,2023-12-29
2,BRK-A,2023-12-29
3,NFLX,2023-12-29
4,TSLA,2023-12-29
5,WMT,2023-12-29


In [27]:
# df_performance por defecto
df_performance = df_activos.copy()
df_performance['CLUSTER'] = n_clusters
df_performance['N_ITERACIONES'] = 0
df_performance['INERTIA'] = float('inf')
df_performance = df_performance.merge(df_date_max, on = 'SIMBOLO', how = 'left')

df_performance_all = pd.DataFrame()
if 'df_performance.pkl' in os.listdir(f'{cofre0}GAN/Clusters/'):
    df_performance_all = pickle_act(f'{cofre0}GAN/Clusters/df_performance')
    df_performance_all['DATE_MAX_TRAIN'] = pd.to_datetime(df_performance_all['DATE_MAX_TRAIN'])

df_performance_all = pd.concat([df_performance_all, df_performance], axis = 0).reset_index(drop = True)
df_performance_all = df_performance_all.drop_duplicates(subset = ['SIMBOLO', 'CLUSTER', 'DATE_MAX_TRAIN']).reset_index(drop = True) # Se prioriza en lo existente, la data por simbolo - cluster - DATE_MAX_TRAIN
df_performance_all = df_performance_all[['SIMBOLO', 'NOMBRE', 'CLUSTER', 'DATE_MAX_TRAIN', 'N_ITERACIONES', 'INERTIA']] # ordenamiento de los campos
df_performance_all

,SIMBOLO,NOMBRE,CLUSTER,DATE_MAX_TRAIN,N_ITERACIONES,INERTIA
0,BTC-USD,Bitcoin,10,2023-12-31,13000,0.131344
1,AAPL,Apple Inc.,10,2023-12-29,37500,0.070594
2,WMT,Walmart,10,2023-12-29,37500,0.044454
3,NFLX,"Netflix, Inc.",10,2023-12-29,37500,0.112262
4,TSLA,Tesla,10,2023-12-29,37500,0.115660
5,AMZN,Amazon,10,2023-12-29,37500,0.110774
6,BRK-A,Berkshire Hathaway,10,2023-12-29,37500,0.041113


In [28]:
df_date_max['IN'] = True
df_date_max

,SIMBOLO,DATE_MAX_TRAIN,IN
0,AAPL,2023-12-29,True
1,AMZN,2023-12-29,True
2,BRK-A,2023-12-29,True
3,NFLX,2023-12-29,True
4,TSLA,2023-12-29,True
5,WMT,2023-12-29,True


In [29]:
df_performance_filtrado = df_performance_all[df_performance_all['CLUSTER'] == n_clusters].reset_index(drop = True) # que tenga el número de clusters requeridos
#df_performance_filtrado = df_performance_filtrado[df_performance_filtrado['SIMBOLO'].isin(df_activos['SIMBOLO'])].reset_index(drop = True) # que esté en la lista de activos
df_performance_filtrado = df_performance_filtrado.merge(df_date_max, on = ['SIMBOLO', 'DATE_MAX_TRAIN'], how = 'left') # esto ya filtra los simbolos seleccionados
df_performance_filtrado['IN'] = df_performance_filtrado['IN'].fillna(False)
df_performance_filtrado = df_performance_filtrado[df_performance_filtrado['IN']].reset_index(drop = True) # que tenga la fecha máxima de entrenamiento
if reiniciar_clusters:
    df_performance_filtrado['N_ITERACIONES'] = 0
df_performance_filtrado = df_performance_filtrado[df_performance_filtrado['N_ITERACIONES'] < n_iteraciones].reset_index(drop = True) # que tenga menos iteraciones de las requeridas
df_performance_filtrado = df_performance_filtrado.drop(columns = ['IN']).reset_index(drop = True)
df_performance_filtrado

,SIMBOLO,NOMBRE,CLUSTER,DATE_MAX_TRAIN,N_ITERACIONES,INERTIA
0,AAPL,Apple Inc.,10,2023-12-29,37500,0.070594
1,WMT,Walmart,10,2023-12-29,37500,0.044454
2,NFLX,"Netflix, Inc.",10,2023-12-29,37500,0.112262
3,TSLA,Tesla,10,2023-12-29,37500,0.115660
4,AMZN,Amazon,10,2023-12-29,37500,0.110774
5,BRK-A,Berkshire Hathaway,10,2023-12-29,37500,0.041113


## Función de ejecución

In [30]:
from sklearn.cluster import KMeans # https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

# Función generar cluster
def generar_clusters(valor, name, df_train, best_inertia, n_clusters = 10, n_iteraciones = 50, max_iter = 500):

    df_train_valor = df_train[df_train['VALOR'] == valor].reset_index(drop = True).copy()
    
    print(f'\n\n\n\n\n\n VALOR: {valor}')
    display('df_train_valor', df_train_valor.head())
    
    # Hacer KMeans con todos los campos excepto (VALOR, DATE, Y). Usar DATE como etiqueta
    X = df_train_valor.drop(columns = ['VALOR', 'DATE', 'Y'])
    y = df_train_valor['DATE']
    
    # Hacer cluster 
    kmeans = KMeans(n_clusters = n_clusters, max_iter = max_iter, verbose = 0,  tol = 10 ** -10, n_init = n_iteraciones) 
    # n_init: Cuantos centroides distintos aleatorios se ejecutan / max_iter: Pasa una iteración (dentro de n_init) cuantas veces itera el algorítmo hasta la convergencia (que no exista movimiento de centroides)
    kmeans.fit(X)
    inertia = kmeans.inertia_ / len(X)
    print('error (intertia)', inertia)
    
    if inertia < best_inertia:
        
        print(f'Mejor modelo {name} new in - best in (antes) - ratio', inertia, best_inertia, inertia / best_inertia)
        best_inertia = inertia
            
        # cuenta cuantos labels hay por cluster
        df_labels = pd.DataFrame({'LABEL': kmeans.labels_})
        df_labels['COUNT'] = 1
        df_labels = df_labels.groupby('LABEL').sum().reset_index()
        display('df_labels', df_labels)
        
        # Se guardan los registros
        df_train_valor['CLUSTER'] = kmeans.labels_
        
        # Se guardan los centroides (después servirán para asignar clusters a nuevos registros)
        df_centroides = pd.DataFrame(kmeans.cluster_centers_, columns = X.columns)
        for c in df_centroides.columns:
            df_centroides = df_centroides.rename(columns = {c: f'CENTROIDE_{c}'})

        display('df_centroides', df_centroides)
        
        print('types',type(df_centroides), type(df_train_valor))
        pickle_act(f'{cofre0}GAN/Clusters/centroides/{name}', df_centroides, 'save')
        pickle_act(f'{cofre0}GAN/Clusters/train_valor/{name}', df_train_valor, 'save')
    
    else:
        # Se rescatan directamente de la mejoropción que existía
        df_centroides = pickle_act(f'{cofre0}GAN/Clusters/centroides/{name}')
        df_train_valor = pickle_act(f'{cofre0}GAN/Clusters/train_valor/{name}')
    
    return df_train_valor, df_centroides, best_inertia


## Proceso de ejecución por tuplas

In [31]:
for i in range(len(df_performance_filtrado)):
    simbolo, nombre, cluster, date_max, n_iteraciones_i, inertia_i = df_performance_filtrado.loc[i]
    name = f'{simbolo}_{cluster}_{str(date_max)[:10]}'
    
    df_train_valor, df_centroides, best_inertia = generar_clusters(simbolo, name, df_train, inertia_i, n_clusters, n_iteraciones - n_iteraciones_i, max_iter) # Se ejecuta la diferencia de iteraciones
    
    df_performance_all.loc[df_performance_all['SIMBOLO'] == simbolo, 'N_ITERACIONES'] = n_iteraciones
    df_performance_all.loc[df_performance_all['SIMBOLO'] == simbolo, 'INERTIA'] = best_inertia

df_performance_all['DATE_MAX_TRAIN'] = pd.to_datetime(df_performance_all['DATE_MAX_TRAIN'])
df_performance_all_respaldo = pd.DataFrame()
if 'df_performance.pkl' in os.listdir(f'{cofre0}GAN/Clusters/'):
    df_performance_all_respaldo = pickle_act(f'{cofre0}GAN/Clusters/df_performance')
    df_performance_all_respaldo['DATE_MAX_TRAIN'] = pd.to_datetime(df_performance_all_respaldo['DATE_MAX_TRAIN'])
    
df_performance_all_respaldo = pd.concat([df_performance_all, df_performance_all_respaldo], axis = 0).reset_index(drop = True)
df_performance_all_respaldo = df_performance_all_respaldo.drop_duplicates(subset = ['SIMBOLO', 'NOMBRE', 'CLUSTER', 'DATE_MAX_TRAIN']).reset_index(drop = True) # Se prioriza en lo existente, la data por simbolo - cluster - DATE_MAX_TRAIN
df_performance_all_respaldo = df_performance_all_respaldo[['SIMBOLO', 'NOMBRE', 'CLUSTER', 'DATE_MAX_TRAIN', 'N_ITERACIONES', 'INERTIA']]
pickle_act(f'{cofre0}GAN/Clusters/df_performance', df_performance_all_respaldo, 'save')







 VALOR: AAPL


'df_train_valor'

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
0,AAPL,1980-12-12,0.750195,0.000000,0.0,0.490819,0.514913,0.000000,0.0,0.526280,...,0.372791,0.000000,0.000000,0.851575,0.0,0.847774,0.000000,0.000000,0.0,0.000000
1,AAPL,1980-12-15,0.750195,0.000382,0.0,0.490819,0.514913,0.000386,0.0,0.526280,...,0.372791,0.000383,0.000382,0.851575,0.0,0.847774,0.000384,0.000401,0.0,-0.052171
2,AAPL,1980-12-16,0.495418,0.000368,0.0,0.490819,0.336816,0.000382,0.0,0.433282,...,0.372791,0.000373,0.000370,0.738120,0.0,0.753043,0.000378,0.000400,0.0,-0.073398
3,AAPL,1980-12-17,0.785880,0.000345,0.0,0.490819,0.315077,0.000373,0.0,0.410579,...,0.372791,0.000354,0.000349,0.867465,0.0,0.861042,0.000366,0.000398,0.0,0.024751
4,AAPL,1980-12-18,0.828714,0.000344,0.0,0.490819,0.400375,0.000368,0.0,0.506472,...,0.372791,0.000349,0.000346,0.886540,0.0,0.876968,0.000360,0.000396,0.0,0.028993


error (intertia) 0.07069595393988941






 VALOR: WMT


'df_train_valor'

...

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
0,AMZN,1997-05-15,0.750195,0.000000,0.0,0.490819,0.514913,0.000000,0.0,0.526280,...,0.372791,0.000000,0.000000,0.851575,0.0,0.847774,0.000000,0.000000,0.0,0.000000
1,AMZN,1997-05-16,0.755639,0.000406,0.0,0.490819,0.514913,0.000410,0.0,0.526280,...,0.372791,0.000407,0.000406,0.851575,0.0,0.849798,0.000408,0.000426,0.0,-0.117028
2,AMZN,1997-05-19,0.769910,0.000372,0.0,0.490819,0.115412,0.000401,0.0,0.317670,...,0.372791,0.000383,0.000377,0.851575,0.0,0.855104,0.000394,0.000424,0.0,-0.012040
3,AMZN,1997-05-20,0.752660,0.000359,0.0,0.490819,0.222931,0.000392,0.0,0.429958,...,0.372791,0.000369,0.000364,0.852672,0.0,0.848690,0.000382,0.000421,0.0,-0.042685
4,AMZN,1997-05-21,0.759777,0.000345,0.0,0.490819,0.266812,0.000382,0.0,0.442098,...,0.372791,0.000354,0.000349,0.830726,0.0,0.851336,0.000370,0.000418,0.0,-0.127392


error (intertia) 0.11111497043267972






 VALOR: BRK-A


'df_train_valor'

,VALOR,DATE,Fibonacci_300_sombra,Suavizamiento_Exponencial_0.7_Close,Media_Movil_300_Close,Media_Movil_4_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.2_Close,Media_Movil_50_Close,Suavizamiento_Exponencial_0.6_rendimiento,...,Media_Movil_120_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.6_Close,Fibonacci_80_sombra,Media_Movil_5_Close,Fibonacci_50_sombra,Suavizamiento_Exponencial_0.3_Close,Suavizamiento_Exponencial_0.05_Close,Media_Movil_30_Close,Y
0,BRK-A,1980-03-17,0.750195,0.00000,0.0,0.490819,0.514913,0.000000,0.0,0.52628,...,0.372791,0.000000,0.000000,0.851575,0.0,0.847774,0.000000,0.000000,0.0,0.0
1,BRK-A,1980-03-18,0.750195,0.00036,0.0,0.490819,0.514913,0.000365,0.0,0.52628,...,0.372791,0.000361,0.000361,0.851575,0.0,0.847774,0.000363,0.000379,0.0,0.0
2,BRK-A,1980-03-19,0.750195,0.00036,0.0,0.490819,0.514913,0.000365,0.0,0.52628,...,0.372791,0.000361,0.000361,0.851575,0.0,0.847774,0.000363,0.000379,0.0,0.0
3,BRK-A,1980-03-20,0.750195,0.00036,0.0,0.490819,0.514913,0.000365,0.0,0.52628,...,0.372791,0.000361,0.000361,0.851575,0.0,0.847774,0.000363,0.000379,0.0,0.0
4,BRK-A,1980-03-21,0.750195,0.00036,0.0,0.490819,0.514913,0.000365,0.0,0.52628,...,0.372791,0.000361,0.000361,0.851575,0.0,0.847774,0.000363,0.000379,0.0,0.0


error (intertia) 0.041160928588093205


# 5. GAN

In [32]:
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

In [33]:
# Solo visualización
verbose_gan = True # Visualización sobre desempeño de GAN
loss_plot = False #True

In [34]:
epochs_gan = 2000 # epochs generados por GAN
n_samples = 5 # sampleos requeridos en total (iteraciones)...si un sampleo mejora, pasará a ser el Sintético principal

num_rows = 1000 # cantidad de filas en las que la GAN, samplea valores, que luego son agrupados en el histograma
bins = 30 # Intervalos del histograma generado

reiniciar_gans = False # Si se dejan todas las iteraciones en 0


In [35]:
"""
SIMBOLO	NOMBRE	CLUSTER	DATE_MAX_TRAIN	N_ITERACIONES	INERTIA
0	BTC-USD	Bitcoin	10	2023-12-31	3500	0.131400
1	AAPL	Apple Inc.	10	2023-12-29	3500	0.070618
2	WMT	Walmart	10	2023-12-29	3500	0.044470
3	NFLX	Netflix, Inc.	10	2023-12-29	3500	0.112307
4	TSLA	Tesla	10	2023-12-29	3500	0.115710
5	AMZN	Amazon	10	2023-12-29	3500	0.110815
6	BRK-A	Berkshire Hathaway	10	2023-12-29	3500	0.041310
"""

'\nSIMBOLO\tNOMBRE\tCLUSTER\tDATE_MAX_TRAIN\tN_ITERACIONES\tINERTIA\n0\tBTC-USD\tBitcoin\t10\t2023-12-31\t3500\t0.131400\n1\tAAPL\tApple Inc.\t10\t2023-12-29\t3500\t0.070618\n2\tWMT\tWalmart\t10\t2023-12-29\t3500\t0.044470\n3\tNFLX\tNetflix, Inc.\t10\t2023-12-29\t3500\t0.112307\n4\tTSLA\tTesla\t10\t2023-12-29\t3500\t0.115710\n5\tAMZN\tAmazon\t10\t2023-12-29\t3500\t0.110815\n6\tBRK-A\tBerkshire Hathaway\t10\t2023-12-29\t3500\t0.041310\n'

In [36]:
n_iteraciones = int(0.8 * delta_days)
n_iteraciones

21

In [37]:
# df_conf_gans por defecto
df_conf_gans = df_activos.copy()
df_conf_gans = df_conf_gans.merge(df_date_max[['SIMBOLO', 'DATE_MAX_TRAIN']], on = 'SIMBOLO', how = 'left')
df_conf_gans['N_CLUSTERS'], df_conf_gans['N_BINS'] = n_clusters, bins
df_conf_gans['N_SAMPLES'], df_conf_gans['BEST_ECM'] = 0, float('inf') # cantidad de iteraciones y medida de performance
df_clusters = pd.DataFrame({'CLUSTER': [i for i in range(n_clusters)]})
df_conf_gans = df_conf_gans.merge(df_clusters, how = 'cross')

# Se lee el df de respaldo si existe
df_conf_gans_all = pd.DataFrame()
if 'df_conf_gans.pkl' in os.listdir(f'{cofre0}GAN/Synt/'): # Synt de Sintético
    df_performance_all = pickle_act(f'{cofre0}GAN/Synt/df_conf_gans')
    df_performance_all['DATE_MAX_TRAIN'] = pd.to_datetime(df_performance_all['DATE_MAX_TRAIN'])
    
df_conf_gans_all = pd.concat([df_conf_gans_all, df_conf_gans], axis = 0).reset_index(drop = True) 
df_conf_gans_all = df_conf_gans_all.drop_duplicates(subset = ['SIMBOLO', 'DATE_MAX_TRAIN', 'N_CLUSTERS', 'CLUSTER', 'N_BINS']).reset_index(drop = True) # Se prioriza en lo existente, la data por simbolo - cluster - DATE_MAX_TRAIN
df_conf_gans_all.head()

,SIMBOLO,NOMBRE,DATE_MAX_TRAIN,N_CLUSTERS,N_BINS,N_SAMPLES,BEST_ECM,CLUSTER
0,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,0
1,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,1
2,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,2
3,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,3
4,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,4


In [38]:
# filtrado

df_conf_gans_filtrado = df_conf_gans_all[df_conf_gans_all['N_CLUSTERS'] == n_clusters].reset_index(drop = True) # que tenga el número de clusters configurado
df_conf_gans_filtrado = df_conf_gans_filtrado.merge(df_date_max, on = ['SIMBOLO', 'DATE_MAX_TRAIN'], how = 'left')
df_conf_gans_filtrado['IN'] = df_conf_gans_filtrado['IN'].fillna(False)

df_conf_gans_filtrado = df_conf_gans_filtrado[df_conf_gans_filtrado['IN']].reset_index(drop = True) # que tenga la fecha máxima de entrenamiento y los simbolos seleccionados
if reiniciar_gans:
    df_conf_gans_filtrado['N_SAMPLES'] = 0  
df_conf_gans_filtrado = df_conf_gans_filtrado[df_conf_gans_filtrado['N_SAMPLES'] < n_iteraciones].reset_index(drop = True) # que tenga menos iteraciones de las requeridas
df_conf_gans_filtrado = df_conf_gans_filtrado.drop(columns = ['IN']).reset_index(drop = True)
df_conf_gans_filtrado.head()

,SIMBOLO,NOMBRE,DATE_MAX_TRAIN,N_CLUSTERS,N_BINS,N_SAMPLES,BEST_ECM,CLUSTER
0,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,0
1,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,1
2,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,2
3,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,3
4,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,4


## Función de ejecución

In [39]:

def ejecutar_GANS_v2(valor, df_train_valor, cluster_id, res_hist, epochs_gan = 1000, verbose_gan = True, loss_plot = True, num_rows = 1000, n_samples = 1, bins = 20, incrementar_epochs = False):
    # valor: simbolo del activo para la ejecución
    # df_train_valor: dataframe con los datos de entrenamiento
    # cluster_id: identificador del cluster en específico para la gan
    # res_hist: diccionario para almacenar los resultados de los histogramas ???...puede ser un df
    # epochs_gan: cantidad de epochs para entrenar la GAN
    # verbose_gan: mostrar información sobre el entrenamiento de la GAN
    # loss_plot: mostrar gráfica de la función de pérdida
    # num_rows: cantidad de filas a generar en la simulación de la GAN -> Esto después se pasa al histograma
    # n_samples: cantidad de muestras sintéticas a generar
    # bins: En cuantos conjuntos se agrupan los num_rows registros
    # incrementar_epochs: si se incrementan los epochs de la GAN en cada iteración (usar esto solo en versiones de prueba)
    
    def plot_histogram(selected_options): # Esta función se debe definir adentro para que funcione bien el gráfico interactivo
        plt.figure(figsize = (10, 6))

        # Graficar histograma de datos reales (siempre presente)
        plt.plot(bin_centers_real, counts_real, linestyle = '-', marker = '', color = 'salmon', label ='Real')
        df_hist_data = pd.DataFrame({'bin_centers': bin_centers_real, 'counts': counts_real / counts_real.sum()}) # se normaliza ya que dependen del ancho de los bins
        df_hist_data['muestra'] = 'Real'
        
        #res_hist[valor][k]['real'] = {'bin_centers': bin_centers_real, 'counts': counts_real}

        # Graficar histogramas sintéticos según selección
        for i, option in enumerate(opciones_sinteticas):  # Iteramos solo sobre los sintéticos
            if option in selected_options:
                counts, bin_edges = np.histogram(synthetic_data_dict[option], bins = bins, density = True)
                #print('info', i, option, counts, bin_edges)
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                #res_hist[valor][k][option] = {'bin_centers': bin_centers, 'counts': counts}
                plt.plot(bin_centers, counts, linestyle = '-', marker = '', color = colores[i], label = option)
                df_hist_data_i = pd.DataFrame({'bin_centers': bin_centers, 'counts': counts / counts.sum()}) # se normaliza ya que dependen del ancho de los bins
                df_hist_data_i['muestra'] = option 
                df_hist_data = pd.concat([df_hist_data, df_hist_data_i])

        plt.xlabel("Rendimiento")
        plt.ylabel("Densidad")
        plt.title(f"Distribución de Rendimientos - Cluster {cluster_id}")
        plt.legend()
        plt.grid(True)
        plt.show()
        res_hist[valor][cluster_id] = df_hist_data
        df_hist_data_ej = df_hist_data.copy()
    
    res_hist[valor] = {}
    df_rend = df_train_valor[df_train_valor['CLUSTER'] == cluster_id].reset_index(drop=True)[['Y']].rename(columns = {'Y': 'rendimiento'})  # Filtrar datos del cluster k
    
    #display('df_rend', df_rend)

    # Calcular histograma de los datos reales una sola vez
    counts_real, bin_edges_real = np.histogram(df_rend, bins = bins, density = True)
    bin_centers_real = (bin_edges_real[:-1] + bin_edges_real[1:]) / 2
    
    df_hist_data = pd.DataFrame({'bin_centers': bin_centers_real, 'counts': counts_real / counts_real.sum()}) # se normaliza ya que dependen del ancho de los bins
    df_hist_data['muestra'] = 'Real'
    
    display('df_hist_data', df_hist_data)
    display('bin_edges_real', bin_edges_real)

    #sys.exit('Continuar el 250402')
    
    # Colores para los histogramas sintéticos (escala entre azul y verde)
    verde = [0, 1, 0]
    azul = [0, 0, 1]
    colores = [[verde[i] * (1 - j) + azul[i] * j for i in range(3)] for j in np.linspace(0, 1, n_samples)]

    # Inicializar diccionario para almacenar datos sintéticos
    synthetic_data_dict = {}
    
    #print(10 * '\ELIMINAR')
    #n_samples = 2

    for i in range(n_samples):  # Para cada muestra sintética
        print('LEN', i, len(df_rend))

        metadata = Metadata.detect_from_dataframe(data = df_rend, table_name = 'value_prices')

        # Inicializar y entrenar CTGAN
        n_epochs = epochs_gan
        if incrementar_epochs:
            n_epochs = epochs_gan * (i + 1)
            
        ctgan = CTGANSynthesizer(metadata, epochs = n_epochs, verbose = verbose_gan)
        ctgan.fit(df_rend)

        # Generar datos sintéticos
        synthetic_data = ctgan.sample(num_rows = num_rows)
        synthetic_data_dict[f'Sintético {i + 1}'] = synthetic_data
        
        display(f'synthetic_data {i + 1}', synthetic_data)

        # Obtener y mostrar la gráfica de la función de pérdida si está habilitada
        if loss_plot:
            fig = ctgan.get_loss_values_plot()
            fig.show()

        #sys.exit('Continua el 250402... quizas se pueden ordenar de < a > los registros y contrastarlos con los reales ordenados de < a > para calcular el ECM')
    # Widget de selección para filtrar qué histogramas sintéticos se mostrarán
    opciones_sinteticas = [f'Sintético {i + 1}' for i in range(n_samples)]
    selector = widgets.SelectMultiple(
        options=opciones_sinteticas,
        value=tuple(opciones_sinteticas),  # Por defecto, se seleccionan todos los sintéticos
        description="Histogramas"
    )

    # Crear interfaz interactiva
    display(widgets.interactive(plot_histogram, selected_options = selector))
    #res_hist[valor][cluster_id] = df_hist_data
    #print('Fin: Ejecucion un solo cluster')
    #break
    
    # Descubrir si alguna de las ejecuciones, mejora el ECM 
    sys.exit('Continuar 250402 buscar ANCLA B2...determinar bien que es lo que se exporta en esta iteración')
    
    display('df_hist_data', df_hist_data)
    return res_hist




## Proceso de ejecución por tuplas

In [40]:
loss_plot = False

In [41]:
num_rows

1000

In [42]:
df_conf_gans_filtrado

,SIMBOLO,NOMBRE,DATE_MAX_TRAIN,N_CLUSTERS,N_BINS,N_SAMPLES,BEST_ECM,CLUSTER
0,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,0
1,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,1
2,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,2
3,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,3
4,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,4
5,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,5
6,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,6
7,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,7
8,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,8
9,NFLX,"Netflix, Inc.",2023-12-29,10,30,0,inf,9


In [43]:
res_hist = {}
for i in range(len(df_conf_gans_filtrado)):
    simbolo, nombre, date_max, n_clusters, n_bins, n_samples, best_ecm, cluster_id = df_conf_gans_filtrado.loc[i]
    date_max = str(date_max)[:10]
    print(simbolo, nombre, date_max, n_clusters, n_bins, n_samples, best_ecm, cluster_id)
    name = f'{simbolo}_{n_clusters}_{date_max}' 
    df_train_valor = pickle_act(f'{cofre0}GAN/Clusters/train_valor/{name}')

    # ejecucion
    res_hist = ejecutar_GANS_v2(valor, df_train_valor, cluster_id, res_hist, epochs_gan = epochs_gan, verbose_gan = verbose_gan, loss_plot = loss_plot, num_rows = num_rows, n_samples = n_iteraciones - n_samples) # n_bins = n_iteraciones - n_bins == ejecuciones = lo requerido - lo que hay hasta el momento
    
    sys.exit('cont 250402')

NFLX Netflix, Inc. 2023-12-29 10 30 0 inf 0


'df_hist_data'

,bin_centers,counts,muestra
0,-0.079105,0.003650,Real
1,-0.069095,0.007299,Real
2,-0.059085,0.000000,Real
3,-0.049075,0.021898,Real
4,-0.039065,0.021898,Real
5,-0.029056,0.062044,Real
6,-0.019046,0.124088,Real
7,-0.009036,0.135036,Real
8,0.000974,0.226277,Real
9,0.010984,0.175182,Real


'bin_edges_real'

array([-0.08410977, -0.07409992, -0.06409007, -0.05408023, -0.04407038,
       -0.03406053, -0.02405069, -0.01404084, -0.00403099,  0.00597886,
        0.0159887 ,  0.02599855,  0.0360084 ,  0.04601824,  0.05602809,
        0.06603794,  0.07604778,  0.08605763,  0.09606748,  0.10607732,
        0.11608717])

...

Gen. (-0.68) | Discrim. (-0.01): 100%|██████████| 2000/2000 [01:28<00:00, 22.63it/s]


'synthetic_data 21'

,rendimiento
0,0.023565
1,-0.016621
2,-0.048239
3,0.028239
4,-0.018772
...,...
995,0.002680
996,-0.009016
997,-0.021017
998,0.026138


interactive(children=(SelectMultiple(description='Histogramas', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 1…

SystemExit: Continuar 250402 buscar ANCLA B2...determinar bien que es lo que se exporta en esta iteración

In [ ]:
# ANCLA B2
df_hist_data_ej = res_hist['TSLA'][0]
df_hist_data_ej.head()

In [ ]:
best_ecm = float('inf')
lista_muestras = list(set(df_hist_data_ej['muestra'].unique()) - {'Real'})
print(lista_muestras)
for muestra in lista_muestras:
    #if i == 0:
    #    continue
    res_hist_cluster_sintetico = df_hist_data_ej[df_hist_data_ej['muestra'].isin([muestra, 'Real'])]
    #display(res_hist_cluster_sintetico)
    # pivot
    res_hist_cluster_sintetico_pivot = res_hist_cluster_sintetico.pivot_table(index = 'bin_centers', columns = 'muestra', values = 'counts').reset_index()
    #display(res_hist_cluster_sintetico_pivot)
    
    for m in [muestra, 'Real']: # interpolación, usando bin center como index (como eje x)
        res_hist_cluster_sintetico_pivot[m] = res_hist_cluster_sintetico_pivot.set_index('bin_centers')[m].interpolate(method = 'index').reset_index()[m]
    
    res_hist_cluster_sintetico_pivot = res_hist_cluster_sintetico_pivot[(res_hist_cluster_sintetico_pivot['Real'].notna()) & (res_hist_cluster_sintetico_pivot[muestra].notna())]
    ecm = ((res_hist_cluster_sintetico_pivot['Real'] - res_hist_cluster_sintetico_pivot[muestra]) ** 2).mean()
    #display(res_hist_cluster_sintetico_pivot)
    print(ecm)
    
    if ecm < best_ecm:
        best_ecm = ecm

print('best', best_ecm)

In [ ]:
sys.exit('OK hasta acá en revisión ágil (250326)')

# La misma idea, comenzar con un df_configuracion_gans con la performance de las combinaciones
# campos SIMBOLO / NOMBRE / DATE_MAX_TRAIN / 
# CLUSTER / bins / n_samples / best_ecm -> Cada combinación (subset) es SIMBOLO / NOMBRE / DATE_MAX_TRAIN / CLUSTER / bins

In [ ]:

def ejecutar_GANS(valor, df_train_valor, n_clusters, res_hist, epochs_gan = 1000, verbose_gan = True, loss_plot = True, num_rows = 1000, n_samples = 1, bins = 20, incrementar_epochs = False):
    res_hist[valor] = {}
    for k in range(n_clusters):
        #res_hist[valor][k] = {}
        df_rend = df_train_valor[df_train_valor['CLUSTER'] == k].reset_index(drop=True)[['Y']].rename(columns={'Y': 'rendimiento'})  # Filtrar datos del cluster k
        
        display('df_rend', df_rend)

        # Calcular histograma de los datos reales una sola vez
        counts_real, bin_edges_real = np.histogram(df_rend, bins = bins, density = True)
        bin_centers_real = (bin_edges_real[:-1] + bin_edges_real[1:]) / 2

        # Colores para los histogramas sintéticos (escala entre azul y verde)
        verde = [0, 1, 0]
        azul = [0, 0, 1]
        colores = [[verde[i] * (1 - j) + azul[i] * j for i in range(3)] for j in np.linspace(0, 1, n_samples)]

        # Inicializar diccionario para almacenar datos sintéticos
        synthetic_data_dict = {}
        
        def plot_histogram(selected_options):
            plt.figure(figsize = (10, 6))

            # Graficar histograma de datos reales (siempre presente)
            counts_real, bin_edges_real = np.histogram(df_rend, bins = bins, density = True)
            bin_centers_real = (bin_edges_real[:-1] + bin_edges_real[1:])
        
            plt.plot(bin_centers_real, counts_real, linestyle = '-', marker = '', color = 'salmon', label ='Real')
            df_hist_data = pd.DataFrame({'bin_centers': bin_centers_real, 'counts': counts_real / counts_real.sum()}) # se normaliza ya que dependen del ancho de los bins
            df_hist_data['muestra'] = 'Real'
            
            #res_hist[valor][k]['real'] = {'bin_centers': bin_centers_real, 'counts': counts_real}

            # Graficar histogramas sintéticos según selección
            for i, option in enumerate(opciones_sinteticas):  # Iteramos solo sobre los sintéticos
                if option in selected_options:
                    counts, bin_edges = np.histogram(synthetic_data_dict[option], bins = bins, density = True)
                    #print('info', i, option, counts, bin_edges)
                    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                    #res_hist[valor][k][option] = {'bin_centers': bin_centers, 'counts': counts}
                    plt.plot(bin_centers, counts, linestyle = '-', marker = '', color = colores[i], label = option)
                    df_hist_data_i = pd.DataFrame({'bin_centers': bin_centers, 'counts': counts / counts.sum()}) # se normaliza ya que dependen del ancho de los bins
                    df_hist_data_i['muestra'] = option 
                    df_hist_data = pd.concat([df_hist_data, df_hist_data_i])

            plt.xlabel("Rendimiento")
            plt.ylabel("Densidad")
            plt.title(f"Distribución de Rendimientos - Cluster {k}")
            plt.legend()
            plt.grid(True)
            plt.show()
    

        for i in range(n_samples):  # Para cada muestra sintética
            print('LEN', i, len(df_rend))

            metadata = Metadata.detect_from_dataframe(data = df_rend, table_name = 'value_prices')

            # Inicializar y entrenar CTGAN
            n_epochs = epochs_gan
            if incrementar_epochs:
                n_epochs = epochs_gan * (i + 1)
                
            ctgan = CTGANSynthesizer(metadata, epochs = n_epochs, verbose = verbose_gan)
            ctgan.fit(df_rend)

            # Generar datos sintéticos
            synthetic_data = ctgan.sample(num_rows = num_rows)
            synthetic_data_dict[f'Sintético {i + 1}'] = synthetic_data

            # Obtener y mostrar la gráfica de la función de pérdida si está habilitada
            if loss_plot:
                fig = ctgan.get_loss_values_plot()
                fig.show()

        # Widget de selección para filtrar qué histogramas sintéticos se mostrarán
        opciones_sinteticas = [f'Sintético {i + 1}' for i in range(n_samples)]
        selector = widgets.SelectMultiple(
            options=opciones_sinteticas,
            value=tuple(opciones_sinteticas),  # Por defecto, se seleccionan todos los sintéticos
            description="Histogramas"
        )

        # Crear interfaz interactiva
        display(widgets.interactive(plot_histogram, selected_options = selector))
        res_hist[valor][k] = df_hist_data
        #print('Fin: Ejecucion un solo cluster')
        #break
    return res_hist





In [ ]:
n_samples

In [ ]:
res_hist = {}
for valor in df_activos['SIMBOLO'].unique():
    print(valor)
    res_hist = ejecutar_GANS(valor, dic_info_clusters[valor]['df_train_valor'], n_clusters, res_hist, epochs_gan = epochs_gan, verbose_gan = verbose_gan, loss_plot = loss_plot, num_rows = num_rows, n_samples = n_samples)
    #sys.exit('Continuar acá el 250310')

# Elección de mejor sintético

In [ ]:
res_hist

In [ ]:
res_hist

res_final = {}
# Elección del mejor sintético
for valor in res_hist:
    print(valor)
    res_hist_valor = res_hist[valor]
    res_final[valor] = {}
    for cluster in res_hist_valor:
        res_hist_cluster = res_hist_valor[cluster]
        display(res_hist_cluster)
        
        best_ecm = float('inf')
        for i in range(n_samples):
            #if i == 0:
            #    continue
            res_hist_cluster_sintetico = res_hist_cluster[res_hist_cluster['muestra'].isin([f'Sintético {i + 1}', 'Real'])]
            #display(res_hist_cluster_sintetico)
            # pivot
            res_hist_cluster_sintetico_pivot = res_hist_cluster_sintetico.pivot_table(index = 'bin_centers', columns = 'muestra', values = 'counts').reset_index()
            #display(res_hist_cluster_sintetico_pivot)
            
            for m in [f'Sintético {i + 1}', 'Real']: # interpolación, usando bin center como index (como eje x)
                res_hist_cluster_sintetico_pivot[m] = res_hist_cluster_sintetico_pivot.set_index('bin_centers')[m].interpolate(method = 'index').reset_index()[m]
            
            res_hist_cluster_sintetico_pivot = res_hist_cluster_sintetico_pivot[(res_hist_cluster_sintetico_pivot['Real'].notna()) & (res_hist_cluster_sintetico_pivot[f'Sintético {i + 1}'].notna())]
            ecm = ((res_hist_cluster_sintetico_pivot['Real'] - res_hist_cluster_sintetico_pivot[f'Sintético {i + 1}']) ** 2).mean()
            #display(res_hist_cluster_sintetico_pivot)
            #print(ecm)
            
            if ecm < best_ecm:
                best_ecm = ecm
                res_final[valor][cluster] = {'sintetico': i + 1, 'ecm': ecm, 'data': res_hist_cluster_sintetico}
            #sys.exit('elegir mejor ecm')
        

#df

# Continuar con los mejores sintéticos

In [ ]:
sys.exit('Continuar acá abajo 250325')

In [ ]:
# ver https://chatgpt.com/c/67e3019c-78cc-8008-aa37-fbf838c9f237 para covarianzas

In [ ]:
dic_info_clusters['BTC-USD']['df_train_valor']

In [ ]:
res_final

In [ ]:
sys.exit('Quitar break en "Fin: Ejecucion un solo cluster"')

In [ ]:
res_hist_cluster.counts.max()

In [ ]:
#res_hist: valor -> cluster -> muestra (epoch) -> bin_centers, counts

# Vista de los clusters

In [ ]:
ver_clusters = False

In [ ]:
for valor in df_activos['SIMBOLO'].unique():
    if not ver_clusters:
        break
    
    df_train_valor= dic_info_clusters[valor]['df_train_valor']
    df_clusters = df_train_valor[['VALOR', 'DATE', 'CLUSTER']]
    df_clusters = df_clusters.merge(df_precios, on = ['VALOR', 'DATE'], how = 'left')

    for k in range(n_clusters):

        print('Salida por ahora...df_train_valor solo tiene un valor (activo)')
        #break
        # Grafica PRECIO vs DATE, en gris claro, excepto donde CLUSTER = k, ese caso graficalo en celeste y con puntos 
        df_clusters_no = df_clusters[df_clusters['CLUSTER'] != k]
        df_clusters_si = df_clusters[df_clusters['CLUSTER'] == k]
        if len(df_clusters_si) == 0:
            continue
        plt.figure(figsize = (15, 5))
        plt.plot(df_clusters_no['DATE'], df_clusters_no['PRECIO'], color = 'lightgray', marker = 'o', linestyle = 'None')
        plt.plot(df_clusters_si['DATE'], df_clusters_si['PRECIO'], color = 'salmon', marker = 'o', linestyle = 'None')
        plt.title(f'{valor} - Cluster {k}')
        plt.show()
    

# Markowitz

In [ ]:
import cvxpy as cp

# Definimos 5 activos inventados
np.random.seed(42)  # Para reproducibilidad
n_assets = 5  # Número de activos

# Retornos esperados de los activos (supuestos)
mu = np.array([0.12, 0.18, 0.15, 0.10, 0.20])

# Matriz de covarianza (simulada aleatoriamente)
Sigma = np.array([
    [0.04, 0.02, 0.01, 0.03, 0.02],
    [0.02, 0.05, 0.02, 0.01, 0.03],
    [0.01, 0.02, 0.03, 0.02, 0.01],
    [0.03, 0.01, 0.02, 0.06, 0.02],
    [0.02, 0.03, 0.01, 0.02, 0.05]
])

# Variables de decisión (pesos del portafolio)
w = cp.Variable(n_assets)

# Retorno objetivo del portafolio
mu_target = 0.14  # Retorno deseado

# Función objetivo: Minimizar la varianza
objective = cp.Minimize(cp.quad_form(w, Sigma))

# Restricciones
constraints = [
    cp.sum(w) == 1,  # La suma de pesos debe ser 1
    w @ mu >= mu_target,  # Retorno esperado mínimo
    w >= 0  # No ventas en corto
]

# Resolver el problema
problem = cp.Problem(objective, constraints)
problem.solve()

# Resultados
weights = w.value
print("Pesos óptimos del portafolio:")
for i, weight in enumerate(weights):
    print(f"Activo {i+1}: {weight:.4f}")

# Graficar los pesos del portafolio
plt.figure(figsize=(8, 5))
plt.bar(range(n_assets), weights, tick_label=[f"Activo {i+1}" for i in range(n_assets)])
plt.ylabel("Peso en el portafolio")
plt.xlabel("Activo")
plt.title("Distribución Óptima del Portafolio de Markowitz")
plt.show()


In [ ]:
import cvxpy as cp

# Definimos 5 activos inventados
np.random.seed(42)  # Para reproducibilidad
n_assets = 5  # Número de activos

# Retornos esperados de los activos (supuestos)
mu = np.array([0.12, 0.18, 0.15, 0.10, 0.20])

# Matriz de covarianza (simulada aleatoriamente)
Sigma = np.array([
    [0.04, 0.02, 0.01, 0.03, 0.02],
    [0.02, 0.05, 0.02, 0.01, 0.03],
    [0.01, 0.02, 0.03, 0.02, 0.01],
    [0.03, 0.01, 0.02, 0.06, 0.02],
    [0.02, 0.03, 0.01, 0.02, 0.05]
])

# Variables de decisión (pesos del portafolio)
w = cp.Variable(n_assets)

# Definir el rango de retornos
mu_min = min(mu)
mu_max = max(mu)
mu_targets = np.linspace(mu_min, mu_max, 10)

# Almacenar resultados
portafolios = []

for mu_target in mu_targets:
    # Función objetivo: Minimizar la varianza
    objective = cp.Minimize(cp.quad_form(w, Sigma))

    # Restricciones
    constraints = [
        cp.sum(w) == 1,  # La suma de pesos debe ser 1
        w @ mu >= mu_target,  # Retorno esperado mínimo
        w >= 0  # No ventas en corto
    ]

    # Resolver el problema
    problem = cp.Problem(objective, constraints)
    problem.solve()

    # Guardar resultados
    weights = w.value
    optimal_value = problem.value
    portafolios.append((mu_target, optimal_value, weights))

# Imprimir los resultados iterados
print("Resultados iterados:")
for i, (mu_t, varianza, weights) in enumerate(portafolios):
    print(f"Iteración {i+1}: Retorno objetivo = {mu_t:.4f}, Varianza mínima = {varianza:.6f}")
    for j, weight in enumerate(weights):
        print(f"   Activo {j+1}: {weight:.4f}")
    print("-")

# Graficar la frontera eficiente
retornos = [x[0] for x in portafolios]
varianzas = [x[1] for x in portafolios]
plt.figure(figsize=(8, 5))
plt.plot(varianzas, retornos, marker='o', linestyle='-')
plt.xlabel("Varianza del portafolio")
plt.ylabel("Retorno esperado")
plt.title("Frontera Eficiente del Portafolio de Markowitz")
plt.grid(True)
plt.show()


# Otros

In [ ]:
sys.exit('Otros antiguos')

# Algorítmo macro

In [ ]:
# Se puede configurar la lstm con los siguientes parámetros:

# 1. número de capas
# 2. número de neuronas en cada capa
# 3. función de activación en cada capa
# 4. Dropout entre capas
# 5. Cerrar o no con una capa densa (fully conected)
# 6. Optimizador (adam, etc)
# 7. Usar bidirectional en cada capa
# 8. cambiar funcion de pérdida (loss)
# 9. ajustar batch size y epochs (en el model.fit)
# 10. LSTM o GRU


## EN RESUMEN
# A. Para cada capa: numero de neuronas, actfunct, dropout [0, 1], usar o no bidirectional (0, 1), es LSTM o GRU
# B. Binario 1 o 0 para cerrar con capa densa
# C. Optimizador (model.compile)
# D. Loss function (model.compile)
# E. Batch size (model.fit)
# F. Dummy para cada uno de los inputs disponibles


# Propuesta:

# En cualquier caso, si por primera vez la red tiene n capas, se agregan los campos de la capa n y fillna 0 a todo lo demás (que antes no existía)

# CAMPOS A: n_neurs_capa_0 (int), actfunct_capa_0 (dummy), dropout_capa_0 (float), bidirectional_capa_0 (bin), LSTM_o_GRU_capa_0 [0 LSTM y 1 GRU] (bin), ...., n_neurs_capa_n (int), actfunct_capa_n (dummy), dropout_capa_n (float), bidirectional_capa_n (bin), LSTM_o_GRU_capa_n (bin), cerrar_con_capa_densa (bin), optimizador (dummy), loss_function (dummy), batch_size (int), epochs (int)
# CAMPOS B-E: [B] cierre_capa_densa (bin), [C] optimizador (dummy), [D] loss_function (dummy), [E] batch_size (int)
# Campos F: cada input disponible (dummy)

In [ ]:
# Propuesta de algoritmo

# 0. Comenzar con una arquitectura definida

# while True:
    # 1. Describir la arquitectura definida en la BD arriba para RLM
    # 2. Crear la red neuronal a partir de la descripción de la arquiectura
    # 3. Entrenar la red neuronal (una cantidad definida de epochs)
    # 4. Ocupar el loss de testeo / promedio de loss de testeo para ese tipo de loss registrados en BD_RLM (para "normalizar" por loss) como campo descriptor
    # 5. Agregar en BD_RLM los campos test_error y test_error_normalizado
    # 6. Guardar la red neuronal en BD_RLM
    # 7. Con un learning rate, buscar un nuevo punto y una nueva arquitectura
    # 8. Guardar BD_RLM
    # 9. Si criterio de salida: salir

# Algorítmo

In [ ]:
# All LSTM: https://chatgpt.com/c/66f9ed7b-bf70-8000-89b4-e2fe09ce9a9e
# Testeo en: https://chatgpt.com/c/67229d51-50bc-8000-9860-dd92babe59a9
# Manejo del learning_rate: https://chatgpt.com/c/6737cfa0-7b6c-8000-90bb-d2c33a882766

In [ ]:
from tensorflow.keras.layers import LeakyReLU

# Opciones disponibles
#act_func = {'tanh': 'tanh', 'relu': 'relu', 'sigmoid': 'sigmoid', 'LeakyReLU': LeakyReLU}
act_func = {'tanh': 'tanh', 'relu': 'relu', 'sigmoid': 'sigmoid'}
act_func_all = act_func.copy()
act_func_all['None'] = None # None: Sin act_func
optimizadores = {'adam': 'adam', 'sgd': 'sgd'}
loss_functions = {'mean_squared_error': 'mean_squared_error', 'mean_absolute_error': 'mean_absolute_error'}

In [ ]:
carpeta_LSTM = '../Cofre/LSTM/'

## df_default (inicial)

In [ ]:
# df_default

# ver: https://docs.google.com/spreadsheets/d/1zIreFcaWyc4qwJ1QBv1pYw_fqtZQwzpmjoNlFcPP33Q/edit?gid=0#gid=0

"""
df_default = pd.DataFrame()

df_default['N_capas'] = [1] # A

for i in range(df_default['N_capas'][0]): # Para cada capa
    df_default[f'n_neurs_capa_{i}'] = [120] # B1

    for af in act_func.keys(): # B2
        df_default[f'actfunct_capa_{i}_{af}'] = [0]
    df_default[f'actfunct_capa_{i}_tanh'] = 1

    df_default[f'dropout_capa_{i}'] = [0.2] # B3
    df_default[f'bidirectional_capa_{i}'] = [0] # B4
    df_default[f'LSTM_o_GRU_capa_{i}'] = [0] # B5

df_default['cerrar_con_capa_densa'] = [0] # C1.1
df_default['n_neurs_capa_densa'] = 64 # C1.2
for af in act_func_all.keys(): # C2
    df_default[f'actfunct_capa_densa_{af}'] = [0]
df_default[f'actfunct_capa_densa_tanh'] = 1 # Sin función de activación

print('TO DO: También puede ser sin act_fun a la salida')

for opt in optimizadores: # D
    df_default[f'optimizador_{opt}'] = [0]
df_default['optimizador_adam'] = 1

for loss in loss_functions: # E
    df_default[f'loss_function_{loss}'] = [0]
df_default['loss_function_mean_squared_error'] = 1

df_default['batch_size'] = [32] # F

for _input in lista_inputs: # G
    df_default[_input] = [1]

df_default
"""

In [ ]:

# f: obtener_base_arquitectura
def obtener_base_arquitectura(df_arquitectura):

    N_capas = int(df_arquitectura['N_capas'][0])
    lista_capas = [f'capa_{i}' for i in range(N_capas)]
    
    df_filtro = df_arquitectura.T.reset_index()
    df_filtro.columns = ['NAME', 'VALUE']

    # 1. Capas

    campos_capa = ['n_neurs', 'actfunct', 'dropout', 'bidirectional', 'LSTM_o_GRU']
    all_dict_capas = {}

    for capa in lista_capas: # Para cada capa
        df_filtro_capa = df_filtro[df_filtro['NAME'].str.contains(capa)] # Filtro solo de la capa
        dic_capa = {}
        for c in campos_capa:
            df_c = df_filtro_capa[df_filtro_capa['NAME'].str.contains(c)].reset_index(drop = True) # Filtro del campo en campos_capa
            if c == 'actfunct':
                df_c = df_c[df_c['VALUE'] == 1].reset_index(drop = True)
                
                try:
                    value = df_c['NAME'][0].split('_')[-1]
                except:
                    print('DISPLAYS')
                    display(df_c)
                    display(df_filtro_capa)
                    display(df_filtro)
                    display(df_arquitectura)
                    sys.exit('Error detectado')
            else:
                value = df_c['VALUE'][0]
            dic_capa[c] = value 
        all_dict_capas[capa] = dic_capa

    # 2. Parámetros generales de configuración (que no dependen de la capa)
    df_filtro = df_filtro[~(df_filtro['NAME'].str.contains('capa_') & ~(df_filtro['NAME'].str.contains('capa_densa')))].reset_index(drop = True) # si contiene capa y no contiene capa densa, excluir

    otros_campos = ['N_capas', 'cerrar_con_capa_densa', 'optimizador', 'loss_function', 'batch_size', 'n_neurs_capa_densa', 'actfunct_capa_densa']
    dict_general = {} 
    eliminar_campos = []
    for c in otros_campos:
        
        df_c = df_filtro[df_filtro['NAME'].str.contains(c)].reset_index(drop = True)

        eliminar_campos += list(df_c['NAME'].unique())
        if c in ['optimizador']:
            df_c = df_c[df_c['VALUE'] == 1].reset_index(drop = True)
            value = df_c['NAME'][0].split('_')[-1]
        elif c in ['loss_function']:
            df_c = df_c[df_c['VALUE'] == 1].reset_index(drop = True)
            value = df_c['NAME'][0].split('_')[2:]
            value = '_'.join(value)
        elif c in ['actfunct_capa_densa']:
            df_c = df_c[df_c['VALUE'] == 1].reset_index(drop = True)
            value = df_c['NAME'][0].split('_')[-1]
        else:
            df_c = df_c[df_c['NAME'] == c].reset_index(drop = True)
            value = df_c['VALUE'][0]
            if value % 1 == 0:
                value = int(value)

        dict_general[c] = value
        
    # 3. Inputs
    df_filtro = df_filtro[~df_filtro['NAME'].isin(eliminar_campos)].reset_index(drop = True)
    
    #display('df_filtro0')
    #display(df_filtro)
    df_filtro = df_filtro[df_filtro['VALUE'] == 1].reset_index(drop = True)
    #display('df_filtro1')
    #display(df_filtro)
    lista_inputs = list(df_filtro['NAME'].unique())

    return all_dict_capas, dict_general, lista_inputs 
    

In [ ]:
df_train

In [ ]:
"""
Explicación de ReduceLROnPlateau:
monitor='val_loss': Monitorea la pérdida de validación.
factor=0.5: Reduce la tasa de aprendizaje multiplicándola por 0.5 (puedes ajustar este valor).
patience=5: Espera 5 épocas sin mejora antes de reducir la tasa de aprendizaje.
min_lr=1e-6: La tasa de aprendizaje mínima a la que puede reducirse.
verbose=1: Muestra un mensaje cuando se reduce la tasa de aprendizaje.
"""

In [ ]:
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU # Llevar arriba
from tensorflow.keras.models import Sequential
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras.models import load_model
from tensorflow.keras.callbacks import Callback

In [ ]:

class IncreaseLROnPlateau(Callback):
    def __init__(self, monitor='val_loss', factor=1.5, patience=5, min_lr=1e-6, max_lr=1e-2, verbose=1):
        super(IncreaseLROnPlateau, self).__init__()
        self.monitor = monitor
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.verbose = verbose
        self.wait = 0
        self.best_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get(self.monitor)
        if current_loss is None:
            return

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
        else:
            self.wait += 1

        if self.wait >= self.patience:
            current_lr = self.model.optimizer.learning_rate.numpy()
            new_lr = min(current_lr * self.factor, self.max_lr)
            if current_lr < self.max_lr:
                self.model.optimizer.learning_rate.assign(new_lr)
                if self.verbose:
                    print(f"\nEpoch {epoch + 1}: Aumentando learning rate a {new_lr:.6f}")
            self.wait = 0

In [ ]:
print('Ancla 0 [funcion crear_LSTM] (ya llamada arriba)')
def crear_LSTM(all_dict_capas, dict_general, lista_inputs):

    print(all_dict_capas)
    print(dict_general)
    print(lista_inputs)

    print('Falta normalizar data (X - min) / (max - min)')

    campos_inputs_df = (set(df_train.columns) - {'VALOR', 'DATE', 'Y'}) # Inputs considerados
    set_inputs_declarados = set(lista_inputs)
    set_inputs = campos_inputs_df.intersection(set_inputs_declarados)
    
    input_shape = len(set_inputs)

    # Inicio
    model = Sequential()
    for i, capa in enumerate(all_dict_capas):
        return_sequences = True
        if i == len(all_dict_capas) - 1:
            return_sequences = False # La última capa no tiene return_sequences
        
        n_neurs, actfunct, dropout, bidirectional, LSTM_o_GRU = all_dict_capas[capa].values()
        
        print('CAPA', i, capa, 'n_neurs', n_neurs, 'actfunct', actfunct, 'dropout', dropout, 'bidirectional', bidirectional, 'LSTM_o_GRU', LSTM_o_GRU)
        n_neurs, bidirectional, LSTM_o_GRU = int(n_neurs), bool(bidirectional), bool(LSTM_o_GRU) # int o booleano dependiendo de los casos
        
        # Creación íntegra de la capa completa
        tipo_capa = LSTM
        if LSTM_o_GRU:
            tipo_capa = GRU
        
        if bidirectional:
            model.add(Bidirectional(tipo_capa(units = n_neurs, return_sequences = return_sequences, input_shape = (input_shape, 1), activation = act_func[actfunct])))
        else:
            model.add(tipo_capa(units = n_neurs, return_sequences = return_sequences, input_shape = (input_shape, 1), activation = act_func[actfunct]))

        # Dropout
        if dropout > 0:
            model.add(Dropout(dropout))
        
        #input_shape = n_neurs

    if dict_general['cerrar_con_capa_densa']:
        model.add(Dense(units = dict_general['n_neurs_capa_densa'], activation = act_func_all[dict_general['actfunct_capa_densa']]))
        print('CAPA DENSA [OPT]', dict_general['n_neurs_capa_densa'], act_func_all[dict_general['actfunct_capa_densa']])
        
    model.add(Dense(units = 1, activation = 'tanh')) # Una sola salida (rendimiento final)...tanh para asegurar valores más suavizados
    print('CAPA FINAL', 'tanh',  1)
    
    if dict_general['optimizador'] == 'adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate = 10 ** -3) # , clipnorm = 1) clipnorm, normaliza datos para que gradiente no explote
    elif dict_general['optimizador'] == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate = 10 ** -3)
    else:
        sys.exit(f"Opt no configurado {dict_general['optimizador']}")
    
    model.compile(optimizer = optimizer, loss = loss_functions[dict_general['loss_function']]) # Optimizador, loss function, learning rate

    # diagrama de la arquitecturaa
    
    # muestra un dibujo de la arquitectura
    print(model.summary())
    #sys.exit()
    
    return model, set_inputs

In [ ]:
print('Ancla 1 [Excelente ejemplo de train - testeo]') # Entrainamiento de la red
print('funcion train_LSTM ya llamada arriba')
def train_LSTM(model, set_inputs, dict_general, df_train, epochs = 50, graficar = True, increase = [False, 0.6]):
    X_train = df_train[list(set_inputs)].values # Se crean las matrices X e Y
    Y_train = df_train['Y'].values

    # Configuración del callback para guardar el mejor modelo
    checkpoint = ModelCheckpoint('mejor_modelo.keras', monitor = 'val_loss', save_best_only = True, mode = 'min', verbose = 1)
    
    # Callback para reducir la tasa de aprendizaje si no hay mejora
    if increase[0]:
        cambios_lr = IncreaseLROnPlateau(monitor = 'val_loss', factor = increase[1], patience = 7, min_lr = 1e-10, verbose = 1)
    else:
        cambios_lr = ReduceLROnPlateau(monitor = 'val_loss', factor = increase[1], patience = 7, min_lr = 1e-10, verbose = 1)

    print('Shapes', X_train.shape, Y_train.shape)
    # Ajuste de la red (entrenamiento)
    history = model.fit(X_train, Y_train, epochs = epochs, batch_size = dict_general['batch_size'], validation_split = test_size / (train_size + test_size), verbose = 1, callbacks = [checkpoint, cambios_lr])

    # Ahora, puedes acceder al historial de error de validación así:
    val_loss_per_epoch = history.history['val_loss']
    
    print('val_loss_per_epoch')
    print(val_loss_per_epoch)

    # Si quieres obtener el error de validación de la última época:
    last_val_loss = val_loss_per_epoch[-1]
    print("Error de validación en la última época:", last_val_loss)

    # Graficar la pérdida de entrenamiento y validación para ver la evolución:
    # Hacer un gráfico a la izq y otro a la derecha
    # Crear una figura con 1 fila y 2 columnas
    if graficar:
        fig, axes = plt.subplots(1, 2, figsize = (12, 5))

        # Gráfico de la pérdida de entrenamiento (izquierda)
        axes[0].plot(history.history['loss'], label='Pérdida de Entrenamiento')
        axes[0].set_title('Pérdida de Entrenamiento')
        axes[0].set_ylabel('Pérdida')
        axes[0].set_xlabel('Época')
        axes[0].legend()

        # Gráfico de la pérdida de validación (derecha)
        axes[1].plot(history.history['val_loss'], label='Pérdida de Validación', color='orange')
        axes[1].set_title('Pérdida de Validación')
        axes[1].set_ylabel('Pérdida')
        axes[1].set_xlabel('Época')
        axes[1].legend()

        # Ajustar el layout para evitar solapamiento
        plt.tight_layout()
        plt.show()

    # Usar `mejor_modelo` para hacer predicciones o evaluaciones
    # Cargar el mejor modelo después del entrenamiento
    mejor_modelo_test = load_model('mejor_modelo.keras')
    
    # error de mejor_modelo_test
    
    return mejor_modelo_test, val_loss_per_epoch


def Predict_LSTM(model, set_inputs, df_control):
    X_control = df_control[list(set_inputs)].values
    Y_control = df_control['Y'].values

    loss = model.evaluate(X_control, Y_control)
    print('Loss en control:', loss)

    return loss

In [ ]:
# muestrame cualquier nan en df_train
print('NAN en df_train')
for c in df_train.columns:
    if len(df_train[df_train[c].isna()]) > 0:
        print(c)
        display(df_train[df_train[c].isna()])

In [ ]:
df_train

## Funciones Etapa 4

In [ ]:

# Función categorizar campo (según reglas)
# Basado en https://docs.google.com/spreadsheets/d/1zIreFcaWyc4qwJ1QBv1pYw_fqtZQwzpmjoNlFcPP33Q/edit?gid=0#gid=0

def categorizar_campo(campo_name):

    if campo_name == 'N_capas':
        cat = 'A'

    elif 'actfunct' in campo_name:
        if 'capa_densa' in campo_name:
            cat = 'C2'
        else:
            capa = campo_name.split('_')[2]
            cat = f'B2-{capa}'

    elif 'n_neurs' in campo_name:
        if 'capa_densa' in campo_name:
            cat = 'C1.2'
        else:
            capa = campo_name.split('_')[3]
            cat = f'B1-{capa}'

    elif 'dropout' in campo_name:
        capa = campo_name.split('_')[2]
        cat = f'B3-{capa}'

    elif 'bidirectional' in campo_name:
        capa = campo_name.split('_')[2]
        cat = f'B4-{capa}'

    elif 'LSTM_o_GRU' in campo_name:
        capa = campo_name.split('_')[4]
        cat = f'B5-{capa}'

    elif 'cerrar_con_capa_densa' == campo_name:
        cat = 'C1.1'

    elif 'optimizador' in campo_name:
        cat = 'D'

    elif 'loss_function' in campo_name:
        cat = 'E'

    elif 'batch_size' == campo_name:
        cat = 'F'

    else:
        cat = 'G'

    return cat


def obtener_df_cat(df_arq, cat):

    #print(cat)
    
    df_cat = df_arq[df_arq['CAT'] == cat].reset_index(drop = True)
    
    #display('df_cat V0')
    #display(df_cat)
    
    # 1. min_max_delta, min y max
    min_max_delta, min_value, max_value = df_cat['MIN_MAX_DELTA_EN_UNA_ITER'][0], df_cat['MIN'][0], df_cat['MAX'][0]
    df_cat['DELTA'] = np.where(abs(df_cat['DELTA']) > min_max_delta, np.sign(df_cat['DELTA']) * min_max_delta, df_cat['DELTA']) # Si delta abs supera min_max_delta, acotar
    
    #display(df_cat)
    df_cat['NEW_VALUE'] = df_cat['VALUE'] - df_cat['DELTA']
    
    if len(df_cat[df_cat['NEW_VALUE'].isna()]) > 0:
        print('Revision nan en df cat')
        display(df_cat[df_cat['NEW_VALUE'].isna()])
        
    df_cat['NEW_VALUE'] = df_cat['NEW_VALUE'].fillna(df_cat['DEFAULT_VALUE']) # Fillna 0
    
    # min & max
    df_cat['NEW_VALUE'] = np.where(df_cat['NEW_VALUE'] < min_value, min_value, df_cat['NEW_VALUE'])
    df_cat['NEW_VALUE'] = np.where(df_cat['NEW_VALUE'] > max_value, max_value, df_cat['NEW_VALUE'])
    
    # 2. Naturaleza del valor
    naturaleza = df_cat['NATURALEZA_VALOR'][0]
    if naturaleza in ['int', 'bin']: # Lógica bin queda corregida aqui. Anteriormente fue aotada entre 0 & 1 según min & max, luego acá se asigna al enteo más cercano
        
        df_cat['NEW_VALUE'] = round(df_cat['NEW_VALUE'] + 0.000001).astype(int)
    elif naturaleza == 'float':
        df_cat['NEW_VALUE'] = df_cat['NEW_VALUE'].astype(float)
    
    logica, comentario = df_cat['LÓGICA'][0], df_cat['LÓGICA COMENTARIO'][0]
    
    #print(logica)
    #print(comentario)
    
    if logica == 'max bin': # CAT B2-0, por ejemplo, es B2 para la capa 0
        max_new_value = df_cat['NEW_VALUE'].max()
        df_cats_max = list(df_cat[df_cat['NEW_VALUE'] == max_new_value]['NAME'].unique())
        # Elegir random de los que cumplen
        name_seleccion = np.random.choice(df_cats_max)
        df_cat['NEW_VALUE'] = np.where(df_cat['NAME'] == name_seleccion, 1, 0)
        
        #print('Random cat', name_seleccion)
        #display(df_cat)
    
    return df_cat


def contar_capas_actual(df_arq):
    return len(df_arq[df_arq['CAT'].str.contains('B1')])


def confirmar_capas(df_cat, df_default, df_conf, df_betas, learning_rate_RLM, df_arq):
    N_capas = int(df_cat[df_cat['NAME'] == 'N_capas'].reset_index(drop = True)['NEW_VALUE'][0]) # Cuantas capas requiere la nueva arquitectura
    print('N_capas', N_capas)

    df_capas = df_default[df_default['NAME'].str.contains('0')].reset_index(drop = True) 
    n_capas_origen = contar_capas_actual(df_arq) # Cuantas hay declaradas

    if N_capas == n_capas_origen:
        return df_arq  # No hacer nada, declaración correcta

    elif N_capas < n_capas_origen:
        # Eliminar las capas sobrantes
        df_capas_delete = pd.DataFrame()
        for i in range(N_capas, n_capas_origen):
            df_delete_new = df_capas.copy()
            df_delete_new['NAME'] = df_delete_new['NAME'].str.replace('0', str(i))
            df_capas_delete = pd.concat([df_capas_delete, df_delete_new])
        df_capas_delete['DEL'] = True
        
        df_arq = df_arq.merge(df_capas_delete, on = 'NAME', how = 'left')
        df_arq['DEL'] = df_arq['DEL'].fillna(False)
        df_arq = df_arq[~df_arq['DEL']].reset_index(drop = True)
        df_arq = df_arq.drop(columns = ['DEL'])

    else: # N_capas > n_capas_origen
        
        #print('N_capas > n_capas_origen en confirmar_capas')
        df_capas_add = pd.DataFrame()
        for i in range(n_capas_origen, N_capas):
            df_add_new = df_capas.copy()
            df_add_new['NAME'] = df_add_new['NAME'].str.replace('0', str(i))
            df_capas_add = pd.concat([df_capas_add, df_add_new])  
            
        # Se agregan las capas por default   
        
        #display(df_capas_add)
        df_capas_add['CAT'] = df_capas_add['NAME'].apply(categorizar_campo)
        #display(df_capas_add)
        df_capas_add['CAT_BASE'] = df_capas_add['CAT'].apply(lambda x: x.split('-')[0])
        #display(df_capas_add)
        df_capas_add = df_capas_add.merge(df_conf, on = 'CAT_BASE', how = 'left')
        #display(df_capas_add)
        df_capas_add[['LÓGICA', 'LÓGICA COMENTARIO']] = df_capas_add[['LÓGICA', 'LÓGICA COMENTARIO']].fillna("")
        #display(df_capas_add)

        # Asignar VALUE por defecto si no existe. Si existe, actualizarlo con learning rate y beta

        if 'VALUE' not in df_capas_add.columns:
            df_capas_add['VALUE'] = df_capas_add['DEFAULT_VALUE'] # Este value por default ya cumple las condiciones

        #  Hacer un for CAT unique, y revisar ese subdataframe filtrando CAT, para dejar los valores finales. Considerar LÓGICA y su descripción en lógica comentario

        # Ejecuar descenso del gradiente
        df_capas_add = df_capas_add.merge(df_betas, on = 'NAME', how = 'left')
        df_capas_add['BETA'] = df_capas_add['BETA'].fillna(0)
        
        #display('df_capas_add (evaluar si todos los betas son 0)')
        #print('suma beta', df_capas_add['BETA'].sum())
        #display(df_capas_add)
        df_capas_add['DELTA'] = df_capas_add['BETA'] * learning_rate_RLM
        df_arq = pd.concat([df_arq, df_capas_add]).reset_index(drop = True)      
    
    return df_arq          
    
    
# 4. Ocupar el loss de testeo / promedio de loss de testeo para ese tipo de loss (ej: 'loss_function': 'mean_squared_error', es un tipo de loss) registrados en BD_RLM (para "normalizar" por loss). La RLM debe "predecir" este valor

def update_BD_RLM(BD_RLM, df_arquitectura, best_val_loss, dict_general):
    # Agrega  df_arquitectura, con los valores de ECM test, loss func y ECM_test_norm a BD RLM
    
    #display('update_BD_RLM 0')
    #display(BD_RLM)
    #BD_RLM_new = BD_RLM.iloc[:-1] 
    print('De comprobarse bien, se puede usar BD_RLM y no el new hacia abajo en la función')
    BD_RLM_new = BD_RLM.copy() # De comrobarse bien, se puede usar BD_RLM y no el new hacia abajo en la función
    
    #display('update_BD_RLM 1')
    #display(BD_RLM_new)
    
    df_arquitectura['ECM_test'] = best_val_loss
    df_arquitectura['loss_func'] = dict_general['loss_function']
    BD_RLM_new = pd.concat([BD_RLM_new, df_arquitectura])
    
    #display('update_BD_RLM 2')

    #BD_RLM_new.to_csv(f'BD_RLM.csv', index = False, sep = ';')
    
    #print('Ver csv...aqui correccion con promedios y eliminar lo de abajo')
    
    BD_RLM_new_mean = BD_RLM_new[['loss_func', 'ECM_test']].groupby('loss_func').mean().reset_index().rename(columns = {'ECM_test': 'ECM_test_mean'})
    BD_RLM_new = BD_RLM_new.merge(BD_RLM_new_mean, on = 'loss_func', how = 'left')
    BD_RLM_new['ECM_test_norm'] = BD_RLM_new['ECM_test'] / BD_RLM_new['ECM_test_mean']
    BD_RLM_new = BD_RLM_new.drop(columns = 'ECM_test_mean')
    
    display('BD_RLM_new')
    display(BD_RLM_new)
    BD_RLM_new.to_csv(f'BD_RLM_Revision0.csv', index = False, sep = ';')
    
    """
    BD_RLM_new_lf = BD_RLM_new[BD_RLM_new['loss_func'] == dict_general['loss_function']].reset_index(drop = True)
    ecm_mean = BD_RLM_new_lf['ECM_test'].mean()
    BD_RLM_new = BD_RLM_new.iloc[:-1]
    
    #display('update_BD_RLM 3')
    #display(BD_RLM_new)
    
    display('BD_RLM_new_lf')
    display(BD_RLM_new_lf)
    
    display('df_arquitectura')
    display(df_arquitectura)

    display('ecm_mean')
    display(ecm_mean)
    
    sys.exit('ecmean')
    
    df_arquitectura['ECM_test_norm'] = best_val_loss / ecm_mean
    BD_RLM_new = pd.concat([BD_RLM_new, df_arquitectura])

    #display('update_BD_RLM 4')
    #display(BD_RLM_new)
    """
    
    return BD_RLM_new


# Ejecutar una RLM con BD_RLM_new
def ejecutar_RLM(BD_RLM_new, min_intercept = 10):
    BD_RLM_new_RLM = BD_RLM_new.drop(columns = ['ECM_test', 'loss_func'])
        
    BD_RLM_new_RLM = BD_RLM_new_RLM.fillna(0)
    
    print('ejecutar_RLM')
    display(BD_RLM_new_RLM)

    X, Y = BD_RLM_new_RLM.drop(columns = ['ECM_test_norm']), BD_RLM_new_RLM[['ECM_test_norm']]

    fit_intercept = True # Prueba con intercepto
    if len(X) >= min_intercept:
        fit_intercept = True
    regr = linear_model.LinearRegression(fit_intercept = fit_intercept)
    regr.fit(X, Y)
    betas = list(regr.coef_[0])
    df_betas = pd.DataFrame({'NAME': list(X.columns), 'BETA': betas})
    return df_betas


def generar_nueva_arquitectura(df_arquitectura = None, df_betas = None, learning_rate_RLM = 1000):

    #def arquitectura_limpia(df_arq, learning_rate):
        
        # A partir de un df arquitectura (con value y mov gradiente) (que puede ser un df vacío, n ese caso, se crea df default (primera iteración),
        # Generar un df arquitectura limpio, basado en el contenido de Inputs/POF_CAT.csv
        # df arquitectura es un registro, con varias columnas

    # def arquitectura_limpia

    # PARAMETROS por default
    if df_betas is None:
        df_betas = pd.DataFrame(columns = ['NAME', 'BETA']) # Valor por default en la funcion
    if df_arquitectura is None:
        df_arquitectura = pd.DataFrame() # Valor por default en la funcion
            
    for c in ['ECM_test', 'loss_func', 'ECM_test_norm']: # Limpieza
        if c in df_arquitectura.columns:
            df_arquitectura = df_arquitectura.drop(columns = c)
            
    df_conf = pd.read_csv(f'{carpeta_input}POF_CAT.csv', sep = ';')
    df_default = pd.read_csv(f'{carpeta_input}df_default.csv', sep = ';')
    #display(df_conf)

    default = False
    if len(df_arquitectura) == 0: # Se crea, sin VALUES inicialmente
        default = True
        df_inputs = pd.DataFrame({'NAME': lista_inputs})
        
        #display('df_default-df_inputs')
        #display(df_default)
        #display(df_inputs)
        df_arquitectura = pd.concat([df_default, df_inputs]).reset_index(drop = True)

    else:
        df_arquitectura = df_arquitectura.T.reset_index()
        df_arquitectura.columns = ['NAME', 'VALUE']

    # En este caso, df_default no tiene valores iniciales. Desde aqui hacia abajo se debe seguir una lógica independiente si es default o no, que actualice los valores y cree una nueva arquitectura
    # Notar siempre que si N_capas aumenta, hay que agregar los campos de la nueva capa por default
    # Si N_capas disminuye, automáticamente no serán considerado los valores de la capa sobrante?? Ej, si pasa de 2 a 1, los valores de la capa 2 no deberían considerarse
    df_arquitectura['CAT'] = df_arquitectura['NAME'].apply(categorizar_campo)
    df_arquitectura['CAT_BASE'] = df_arquitectura['CAT'].apply(lambda x: x.split('-')[0])
    df_arquitectura = df_arquitectura.merge(df_conf, on = 'CAT_BASE', how = 'left')
    df_arquitectura[['LÓGICA', 'LÓGICA COMENTARIO']] = df_arquitectura[['LÓGICA', 'LÓGICA COMENTARIO']].fillna("")

    #display(df_arquitectura)
    # Asignar VALUE por defecto si no existe. Si existe, actualizarlo con learning rate y beta

    if 'VALUE' not in df_arquitectura.columns:
        df_arquitectura['VALUE'] = df_arquitectura['DEFAULT_VALUE'] # Este value por default ya cumple las condiciones

    #  Hacer un for CAT unique, y revisar ese subdataframe filtrando CAT, para dejar los valores finales. Considerar LÓGICA y su descripción en lógica comentario

    # Ejecuar descenso del gradiente

    df_arquitectura = df_arquitectura.merge(df_betas, on = 'NAME', how = 'left').reset_index(drop = True)
    df_arquitectura['BETA'] = df_arquitectura['BETA'].fillna(0)
    
    #print('S BETAS', df_arquitectura['BETA'].sum())
    
    if (df_arquitectura['BETA'].sum() == 0) and (not default):
        print('Distorsion')
        df_arquitectura['BETA'] = np.random.uniform(-1, 1, len(df_arquitectura)) # Distorsión
        
    df_arquitectura['DELTA'] = df_arquitectura['BETA'] * learning_rate_RLM
    
    #print('df arq betas')
    #display(df_arquitectura.head(30))
    #display(df_arquitectura.tail(30))

    df_new_arquitectura = pd.DataFrame()

    # Primero, df_cat = obtener_df_cat(df_arq, cat) se ejecuta para A, para conocer el número final de capas de la red

    df_cat = obtener_df_cat(df_arquitectura, 'A')
    df_new_arquitectura = pd.concat([df_new_arquitectura, df_cat])

    # ANCLA 3
    df_arquitectura = confirmar_capas(df_cat, df_default, df_conf, df_betas, learning_rate_RLM, df_arquitectura) # Agrega o quita campos de capas según N_capas nuevo
    
    #df_arquitectura.to_csv(f'df_arquitectura_revision.csv', index = False, sep = ';')
    
    #print(df_arquitectura['CAT'].unique())
    #sys.exit()

    for cat in df_arquitectura['CAT'].unique():
        if cat == "A": # "A" ya fue agregado
            continue
        #print(cat)
        df_cat = obtener_df_cat(df_arquitectura, cat)
        
        #print('cat', cat)
        #display(df_cat)
        df_new_arquitectura = pd.concat([df_new_arquitectura, df_cat])
    
    #sys.exit()
        
    df_new_arquitectura = df_new_arquitectura.reset_index(drop = True)
    
    #display('df_new_arquitectura 0')
    #display(df_new_arquitectura)

    # Retornar esto en la función, es la nueva arquitectura que cumple con todas las condiciones
    df_new_arquitectura_clean = df_new_arquitectura[['NAME', 'NEW_VALUE']].T
    df_new_arquitectura_clean.columns = df_new_arquitectura_clean.iloc[0]
    df_new_arquitectura_clean = df_new_arquitectura_clean.iloc[1:]
    
    #display('df_new_arquitectura 1')
    #display(df_new_arquitectura_clean)
    
    return df_new_arquitectura_clean


# Ejecución

In [ ]:
learning_rate_RLM = 100 # Parámetros
iteraciones_arquitecturas = 10
epochs = 3 # Para cada red neuronal
graficar = True

In [ ]:
# 0. Comenzar con una arquitectura definida

# 0.A: Se lee la BD de RLM
#df_arquitectura = generar_nueva_arquitectura()  # default
BD_RLM = pd.DataFrame()
if 'BD_RLM.pkl' in os.listdir(carpeta_LSTM):
    BD_RLM = pickle_act(f'{carpeta_LSTM}BD_RLM', mode = 'open')


# 1.A: Seleccionar el último registro como arquitectura
if len(BD_RLM) == 0:
    df_arquitectura = generar_nueva_arquitectura() # default
else:
    df_arquitectura = BD_RLM.tail(1).reset_index(drop = True)

for i in range(iteraciones_arquitecturas):
    # 1. Describir la arquitectura definida en la BD arriba para RLM
    
    display('df_arquitectura inicial')
    display(df_arquitectura)
    
    #print('Continuar acá: crear función obtener_base_arquitectura')
    all_dict_capas, dict_general, lista_inputs = obtener_base_arquitectura(df_arquitectura) # Función que obtiene la base de la arquitectura, devuelve el detalle de cada capa en diccionario + otros parametros  # Basicamente, interpreta las dummies y otras cosas
    print('all_dict_capas (diccionario con la configuración de cada capa)', all_dict_capas)
    print('diccionario con la configuración general de la red',  dict_general)
    print('inputs considerados en la red', lista_inputs)

    # 2. Crear la red neuronal a partir de la descripción de la arquitectura
    print('2. Crear red \n\n\n')
    model, set_inputs = crear_LSTM(all_dict_capas, dict_general, lista_inputs) # Función que crea la red neuronal a partir de la descripción de la arquitectura
    # 3. Entrenar la red neuronal (una cantidad definida de epochs)
    print('3. Entrenar red \n\n\n')
    mejor_modelo_test, val_loss_per_epoch = train_LSTM(model, set_inputs, dict_general, df_train, epochs = epochs, graficar = graficar)
    best_val_loss = min(val_loss_per_epoch) # El mejor de la lista que acumula el loss en cada epoch
    
    # 4a. Ocupar el loss de testeo / promedio de loss de testeo para ese tipo de loss (ej: 'loss_function': 'mean_squared_error', es un tipo de loss) registrados en BD_RLM (para "normalizar" por loss). La RLM debe "predecir" este valor
    # 4b. Agregar en BD_RLM los campos test_error y test_error_normalizado (incluido en este subproceso)
    # 4c. Guardar la red neuronal en BD_RLM (no lo veo necesario. Es mejor tener una preselección y, en otro código, entrenar más este conjunto de mejores redes)
    # 4d. Con un learning rate, buscar un nuevo punto y una nueva arquitectura
    
    BD_RLM = update_BD_RLM(BD_RLM, df_arquitectura, best_val_loss, dict_general) # Agrega df arquitectura a BD con valores de ECM y ECM norm    
    df_betas = ejecutar_RLM(BD_RLM) # Se buscan los coeficientes beta (sin intercepto en las primeras iteraciones

    df_arquitectura = generar_nueva_arquitectura(df_arquitectura = df_arquitectura, df_betas = df_betas, learning_rate_RLM = learning_rate_RLM) # Se crea una nueva arquitectura

    # 5. Guardar BD_RLM
    pickle_act(f'{carpeta_LSTM}BD_RLM', BD_RLM, mode = 'save')
    # 6. Si criterio de salida: salir
    
    #BD_RLM = BD_RLM_new.copy()
    #df_arquitectura = df_arquitectura_nueva.copy()
    

for c in BD_RLM.columns:# [['mean_absolute_error']]
    #print(c)
    try:
        BD_RLM[c] = BD_RLM[c].astype(float)
    except:
        #display(BD_RLM[[c]])
        None

BD_RLM.to_csv(f'BD_RLM.csv', index = False, sep = ';', decimal = ',')
display('Ultima arquitectura')
display(df_arquitectura)
sys.exit('Continuar acá: ver url https://chatgpt.com/c/66f9ed7b-bf70-8000-89b4-e2fe09ce9a9e')  
# Ver: # Anexo: Predict y medición



## Fallas

#actfunct = LeakyReLu
# bidirectional = True
# GRU (en vez de LSTM)
# No cerrar con capa densa


In [ ]:
df_betas.BETA.min(), df_betas.BETA.max()

# To do

In [ ]:
# OK 250107 sys.exit('ecm coregido debe actualizarse para toda bd rlm')
# OK 250107 print('Crear primero caso df_default (f: generar_nueva_arquitectura)')

In [ ]:
print('1a. Falta crear un lista_input como lista_inputs inicial. De esta forma lista_inputs son todos los permitidos en algún momento, mientras que lista-inputs_inicial, los que están en el df_default (buscar: # Se crea, sin VALUES inicialmente...aqui debe ir con lista_inputs_inicial')
print('1b. Al ejecutar este proceso, un nuevo input en lista input (que no haya sido visto antes, debe ser integrado en la rrnn actual, más todo lo que esa red ya tenía...Por lo tanto el default inicial será lista inputs)')
print('Ajustar el error cuadratico medio test a una parábola con A > 0 para evaluarcuantos epochs ejecutar y donde hay más potencial')
sys.exit('2a. POF_CAT cambiar la lógica de los inputs de la red, para que exista una cantidad mínima. Ej: El 50% de los inputs al menos debe ser 1') 

In [ ]:
# Guardar el mejor modelo de cada iteración (puede ser en el mismo DF de RLM)...cada iteración es una configuración distinta de red y tiene varios epochs
# Parábola para estimar curva de ECM de testeo, para identificar cuantos epochs ejecutar
# Medir tiempo medio x epoch de cada iteración y considerarlo...pueden habr redes muy buenas, pero si son enormes y se demoran mucho, no son tan buenas
# Elegir los k mejores modelos, cada uno con sus predicciones...El modelo i tendrá una ponderación P_i = H / error de testeo normalizado i (inv proporcional al error norm de testeo)...H es una constante tal que sum(i, P_i) = 1
    # Estimar el valor medio, varianza y covarianza de los valores en, por ejemplo, una semana (en los prox 7 días).....recalibrar markowitz una vez a la semana

# Evaluar con el grupo de control

## PARA GAN

    # Para cada valor, ocupar df_train para "clusterizar" los distintos días de CADA VALOR y obtener una distribuión del rendimiento para cada cluster......para un nuevo día, se asigna a un cluster y se le asigna un sampleo de la distribución GAN asociada a ese cluster

In [ ]:
sys.exit('Continuar acá: ver url https://chatgpt.com/c/66f9ed7b-bf70-8000-89b4-e2fe09ce9a9e')

# Anexo: Predict y medición

In [ ]:
display(df_control.head())

# Prueba predict (llevar a función Predict_LSTM)

X_control = df_control[list(set_inputs)].values
Y_control = df_control['Y'].values

loss = mejor_modelo_test.evaluate(X_control, Y_control)
print('Loss en control:', loss)

# Muestra los resultados
predicciones = mejor_modelo_test.predict(X_control)
df_predicciones = df_control.copy()
df_predicciones['PREDICCION'] = predicciones

df_control['PREDICCION'] = predicciones

# Calcula ECM y EAM

df_control['ERROR'] = df_control['Y'] - df_control['PREDICCION']
df_control['ERROR_ABS'] = abs(df_control['ERROR'])
eam = df_control['ERROR_ABS'].mean()

df_control['ERROR_CUAD'] = df_control['ERROR'] ** 2
ecm = df_control['ERROR_CUAD'].mean()

print('ECM:', ecm, 'EAM:', eam)



# KPIs medición

df_errores = pd.DataFrame()
df_kpis = pd.DataFrame()
df_clasificacion_all = pd.DataFrame()

for i in range(len(df_activos)):
    simbolo, nombre = df_activos.loc[i]
    df_control_i = df_control[df_control['VALOR'] == simbolo].reset_index(drop = True)
    df_control_i = df_control_i.merge(df_precios, on = ['VALOR', 'DATE'], how = 'left')
    df_control_i = df_control_i[['VALOR', 'DATE', 'PRECIO', 'Y', 'PREDICCION']]
    df_control_i = df_control_i.sort_values('DATE').reset_index(drop = True)
    precio_base = df_control_i['PRECIO'][0] / (1 + df_control_i['Y'][0])
    
    df_control_i['CLASIFICACION'] = np.where(df_control_i['Y'] >= 0, np.where(df_control_i['PREDICCION'] >= 0, 'TP', 'FN'), np.where(df_control_i['PREDICCION'] >= 0, 'FP', 'TN'))

    df_clasificacion_i = df_control_i[['CLASIFICACION']]
    df_clasificacion_i['CUENTA'] = 1
    df_clasificacion_i = df_clasificacion_i.groupby('CLASIFICACION').sum().reset_index()
    df_clasificacion_i['VALOR'] = simbolo
    df_clasificacion_all = pd.concat([df_clasificacion_all, df_clasificacion_i], axis = 0)
    
    lista_ops = list(df_clasificacion_i['CLASIFICACION'].unique())
    for c in ['FN', 'FP', 'TN', 'TP']:
        if c not in lista_ops:
            df_clasificacion_i = pd.concat([df_clasificacion_i, pd.DataFrame({'CLASIFICACION': [c], 'CUENTA': [0], 'VALOR': [simbolo]})])

    display(df_clasificacion_i)
    
    accuracy = df_clasificacion_i[df_clasificacion_i['CLASIFICACION'].isin(['TP', 'TN'])]['CUENTA'].sum() / df_clasificacion_i['CUENTA'].sum() # True, sobre todos los demás
    precision = df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'TP']['CUENTA'].values[0] / (df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'TP']['CUENTA'].values[0] + df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'FP']['CUENTA'].values[0]) # True, sobre todos los que se predijeron como True
    recall = df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'TP']['CUENTA'].values[0] / (df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'TP']['CUENTA'].values[0] + df_clasificacion_i[df_clasificacion_i['CLASIFICACION'] == 'FN']['CUENTA'].values[0]) # True, sobre todos los que son True

    df_kpis_new = pd.DataFrame({'VALOR': [simbolo], 'ACCURACY': [accuracy], 'PRECISION': [precision], 'RECALL': [recall]})
    df_kpis = pd.concat([df_kpis, df_kpis_new], axis = 0)

    df_control_i['PRECIO_PREDICCION'] = 0
    df_control_i['PRECIO_PREDICCION'][0] = precio_base * (1 + df_control_i['PREDICCION'][0])
    
    for i in range(1, len(df_control_i)):
        df_control_i['PRECIO_PREDICCION'][i] = df_control_i['PRECIO'][i - 1] * (1 + df_control_i['PREDICCION'][i]) 
    #display(df_control_i)
    
    ecm = ((df_control_i['Y'] - df_control_i['PREDICCION']) ** 2).mean()
    eam = abs(df_control_i['Y'] - df_control_i['PREDICCION']).mean()
    n = len(df_control_i)
    
    df_errores_new = pd.DataFrame({'VALOR': [simbolo], 'ECM': [ecm], 'EAM': [eam], 'N': [n]})
    df_errores = pd.concat([df_errores, df_errores_new], axis = 0)
    
    # plotear PRECIO Y PRECIO_PREDICCION

    #df_control_i = df_control_i[df_control_i['DATE'] <= "2020-01-01"]
    plt.figure(figsize = (12, 6))
    plt.plot(df_control_i['DATE'], df_control_i['Y'], label = 'Rendimiento Real')
    plt.plot(df_control_i['DATE'], df_control_i['PREDICCION'], label = 'Rendimiento Predicción')
    plt.title(f'{nombre} - {simbolo}')
    plt.legend()
    plt.show()


display(df_kpis)
display(df_errores)

ecm_tot = (df_errores['ECM'] * df_errores['N']).sum() / df_errores['N'].sum()
eam_tot = (df_errores['EAM'] * df_errores['N']).sum() / df_errores['N'].sum()

print('ecm_tot', ecm_tot, 'eam_tot', eam_tot)

display(df_clasificacion_all)





In [ ]:
# KPIs agregados (para todos  los valores)
df_clasificacion_all_agr = df_clasificacion_all[['CLASIFICACION', 'CUENTA']].groupby('CLASIFICACION').sum().reset_index()

lista_ops = list(df_clasificacion_all_agr['CLASIFICACION'].unique())
for c in ['FN', 'FP', 'TN', 'TP']:
    if c not in lista_ops:
        df_clasificacion_all_agr = pd.concat([df_clasificacion_all_agr, pd.DataFrame({'CLASIFICACION': [c], 'CUENTA': [0], 'VALOR': ['ALL']})])

display(df_clasificacion_all_agr)

accuracy = df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'].isin(['TP', 'TN'])]['CUENTA'].sum() / df_clasificacion_all_agr['CUENTA'].sum() # True, sobre todos los demás
precision = df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'TP']['CUENTA'].values[0] / (df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'TP']['CUENTA'].values[0] + df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'FP']['CUENTA'].values[0]) # True, sobre todos los que se predijeron como True
recall = df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'TP']['CUENTA'].values[0] / (df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'TP']['CUENTA'].values[0] + df_clasificacion_all_agr[df_clasificacion_all_agr['CLASIFICACION'] == 'FN']['CUENTA'].values[0]) # True, sobre todos los que son True

df_kpis_new = pd.DataFrame({'VALOR': ['ALL'], 'ACCURACY': [accuracy], 'PRECISION': [precision], 'RECALL': [recall]})
df_kpis_new

In [ ]:
print('Primero, generar algorítmo macro')
sys.exit()

# Otros

In [ ]:
sys.exit()

### LSTM Básica (Revisar este ejemplo)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# Generar datos de ejemplo: secuencia de números
data = np.array([i for i in range(1000)], dtype=float)
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, 1))

# Definir el tamaño de las secuencias
timesteps = 10
X = []
y = []

for i in range(len(data) - timesteps):
    X.append(data[i:i + timesteps])
    y.append(data[i + timesteps])

X = np.array(X)
y = np.array(y)

# Redimensionar X para que tenga forma (muestras, timesteps, características)
X = np.reshape(X, (X.shape[0], timesteps, 1))

# Crear el modelo LSTM
model = Sequential()
model.add(LSTM(units=50, return_sequences=False, input_shape=(timesteps, 1)))
model.add(Dense(units=1))

# Compilar el modelo
model.compile(optimizer='adam', loss='mean_squared_error')

# Entrenar el modelo
model.fit(X, y, epochs=10, batch_size=32)

# Predecir utilizando el modelo entrenado
predicciones = model.predict(X)

# Invertir la normalización para ver los valores originales de la predicción
# Volver a darle forma para el escalado inverso
predicciones_reshaped = predicciones.reshape(-1, 1)
predicciones_originales = scaler.inverse_transform(predicciones_reshaped)

# Mostrar primeras 5 predicciones desescaladas
print(predicciones_reshaped[:5])


In [ ]:
predicciones.shape, y.shape

In [ ]:
y.min(), y.max()

In [ ]:
# Graficando ambos vectores
plt.figure(figsize=(12, 6))
plt.plot(predicciones, label="Predicciones")
plt.plot(y, label="Y", alpha=0.75)

plt.title("Comparación de Predicciones Originales y Y")
plt.xlabel("Índice")
plt.ylabel("Valores")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
((predicciones_originales - y) ** 2).mean()

In [ ]:
predicciones_originales.shape

In [ ]:
# Caso con retorno
# 
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed
from sklearn.preprocessing import MinMaxScaler

# Generar datos de ejemplo: secuencia de números
data = np.array([i for i in range(1000)], dtype=float)
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, 1))

# Definir el tamaño de las secuencias
timesteps = 10
X = []
y = []

for i in range(len(data) - timesteps):
    X.append(data[i:i + timesteps])
    y.append(data[i + 1:i + timesteps + 1])  # Desplazar las etiquetas en 1 paso hacia adelante

X = np.array(X)
y = np.array(y)

# Redimensionar X para que tenga forma (muestras, timesteps, características)
X = np.reshape(X, (X.shape[0], timesteps, 1))
y = np.reshape(y, (y.shape[0], timesteps, 1))  # También se da forma a 'y' para cada paso de tiempo

# Crear el modelo LSTM
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(timesteps, 1)))
model.add(TimeDistributed(Dense(units=1)))  # Predicción en cada paso de tiempo

# Compilar el modelo
model.compile(optimizer='adam', loss='mean_squared_error')

# Entrenar el modelo
model.fit(X, y, epochs=10, batch_size=32)

# Predecir utilizando el modelo entrenado
predicciones = model.predict(X)

# Invertir la normalización para ver los valores originales
predicciones_originales = scaler.inverse_transform(predicciones.reshape(-1, 1)).reshape(predicciones.shape)

# Mostrar primeras 5 secuencias de predicciones
print(predicciones_originales[:5])


# Clase Red Neuronal

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from keras.initializers import RandomUniform

In [ ]:
class Callback(tf.keras.callbacks.Callback): # Esta clase impide que en el entrenamiento de la FFNN, se muestren los resultados de cada epoch (model.fit)
    SHOW_NUMBER = 30
    counter = 0
    epoch = 0

    def on_epoch_begin(self, epoch, logs = None):
        self.epoch = epoch

    def on_train_batch_end(self, batch, logs = None):
        if self.counter == self.SHOW_NUMBER or self.epoch == 1:
            None
            #print('Epoch: ' + str(self.epoch) + ' loss: ' + str(logs['loss']))
            if self.epoch > 1:
                self.counter = 0
        self.counter += 1

In [ ]:
def definir_arquitectura(str_arquitectura): # convierte un str de arquitectura a una matriz de arquitectura
    arquitectura = []
    if str_arquitectura == "0":
        return arquitectura
    
    lista_neuronas = str_arquitectura.split(',')

    for neurona in lista_neuronas:
        arquitectura.append([neurona.split('_')[0], int(neurona.split('_')[1])])
    
    return arquitectura

In [ ]:
def generar_matriz_transformacion(lista_representativa, eliminar):
    # Solo permite eliminar filas y columnas
    vector_rep = np.array(lista_representativa)
    #  T es una mariz de ceros, de dimensiones vector_rep.sum() [cantidad de capos compartidos, interseccion] x len(campos_base_x)
    T = np.zeros((vector_rep.sum(), len(vector_rep)))

    i, j = 0, 0
    for k in range(len(vector_rep)):
        if vector_rep[k] == 1:
            T[i, j] = 1
            i, j = i + 1, j + 1
        else:
            j += 1

    if eliminar == 'columnas': # Se devuelve la matriz transpuesta
        T = T.T
    return T


def generar_matriz_transformacion_estructurada(lista_representativa):
    # Permite agregar filas y columnas random
    
    vector_rep = np.array(lista_representativa)
    T = np.zeros((len(vector_rep), vector_rep.sum()))
    T.shape

    i, j = 0, 0
    for k in range(len(vector_rep)):
        if vector_rep[k] == 1:
            T[i, j] = 1
            i, j = i + 1, j + 1
        else:
            for j2 in range(vector_rep.sum()):
                # agregar un random uniform entre -1 y 1
                T[i, j2] = np.random.uniform(-1, 1)
            i += 1

    return T
    

In [ ]:
def estandarizar(df):
    df_estandarizacion = pd.DataFrame()
    for c in list(set(df.columns) - {'VALOR', 'DATE', 'PREDICT'}):
        promedio, desvest = df[c].mean(), df[c].std()
        df_estandarizacion_new = pd.DataFrame({'CAMPO': [c], 'PROMEDIO': [promedio], 'DESVIACION': [desvest]})
        df_estandarizacion = pd.concat([df_estandarizacion, df_estandarizacion_new])
        df[c] = (df[c] - promedio) / desvest
    return df, df_estandarizacion

def normalizar(df):
    df_normalizacion = pd.DataFrame()
    for c in list(set(df.columns) - {'VALOR', 'DATE', 'PREDICT'}):
        n_min, n_max = df[c].min(), df[c].max()
        df_normalizacion_new = pd.DataFrame({'CAMPO': [c], 'MIN': [n_min], 'MAX': [n_max]})
        df_normalizacion = pd.concat([df_normalizacion, df_normalizacion_new])
        df[c] = (df[c] - n_min) / (n_max - n_min)
    return df, df_normalizacion


In [ ]:
class Red_Neuronal():

    def __init__(self, str_arquitectura, campos_input, cofre, seguimiento, output_level, metrica, df_activos, herencia = False, nombre_clase_heredada = None, cambio = None, dic_cambios = None, campos_input_heredados = None): 
    
        # Se guardan en el init, los atributos iniciales del objeto
        self.str_arquitectura = str_arquitectura # arquitectura asociada a la red neuronal 
        #self.nombre = nombre # nombre de la red neuronal
        self.cofre = f'{cofre}Red_Neuronal/' # donde se guardan y rescatan las redes, con su info actualizada
        self.seguimiento = seguimiento
        self.output_level = output_level
        self.metrica = metrica
        self.raw_x = pd.DataFrame({'DATE': [], 'NAME': []})
        #self.campos_input = campos_input
        #self.campos_output = campos_output
        self.dic_data = {}
        self.best_test_loss = float('inf')
        self.herencia = herencia
        self.nombre_clase_heredada = nombre_clase_heredada
        self.cambio = cambio
        self.dic_cambios = dic_cambios
        self.df_activos = df_activos
        
        #print('Activos seleccionados 0')
        #display(self.df_activos)
        
        self.campos_input_heredados = campos_input_heredados
        #self.campos_input_heredados = campos_input
        #self.epochs = epochs
        
        self.campos_output = f'Y_{self.metrica}_{self.output_level}'
        
        self.inicializar(campos_input, self.campos_output)
        self.modo_arquitectura = self.rescatar() # Rescata el objeto (y toda su información) si existe
        
        #print('Activos seleccionados')
        #display(self.df_activos)
        return None
                
    def rescatar(self):
        if f'{self.nombre}.pkl' not in os.listdir(self.cofre):
            #print('Rescatar')
            self.arquitectura = definir_arquitectura(self.str_arquitectura) # Se define la arquitectura con su nomeclatura
            #print('NAME REV')
            #print(self.str_arquitectura, self.arquitectura)
            
            if self.herencia: # Si se puede, se rescatan los datos del objeto heredado
                print('SI HERENCIA FFNN')
                valor_cargado_heredado = pickle_act(f'{self.cofre}{self.nombre_clase_heredada}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos
                for key, value in vars(valor_cargado_heredado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
                    if key in ['herencia', 'nombre_clase_heredada', 'cambio', 'dic_cambios', 'str_arquitectura', 'cofre', 'seguimiento', 'output_level', 'arquitectura', 'campos_input', 'campos_input_heredados', 'df_activos']: # atributos no heredados
                        continue
                    elif key == 'model':
                        setattr(self, 'model_base', value) # Aquí, model se renombra como model base
                    #elif key == 'campos_input':
                    #    setattr(self, 'campos_input_heredados', value)
                    # si se heredan! dic_data, model 
                    setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
                #print('NAME REV2')
                #print(self.str_arquitectura, self.arquitectura)
                
                self.cambiar_ffnn() # El cambio solo existe si hay una herencia, y la nueva red no existe
            else:    
                self.crear_ffnn() # Crea la estructura inicial de la red
            return 'nueva' # Si no se encuentra, no se pueden rescatar los atributos
        
        #print('RESCATE!!')
        valor_cargado = pickle_act(f'{self.cofre}{self.nombre}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos (todos)
        for key, value in vars(valor_cargado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
            if key in ['df_activos']: # atributos no heredados
                continue
            setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
        return 'rescate'
    
    
    def guardar(self):
        pickle_act(f'{self.cofre}{self.nombre}', variable = self, mode = 'save')
        return None
    
    def crear_ffnn(self, loss = 'mean_squared_error', optimizer = 'adam', metrics = ['accuracy']):
        
        dic_metricas = {'Rendimiento': None, 'Varianza': 'relu'}
        # Inicializador con valores entre 0 y 1
        initializador = RandomUniform(minval = 0, maxval = 1)
    
        # Crea un modelo secuencial
        self.model = keras.Sequential()
        
        if len(self.arquitectura) == 0: # Sin hidden layers
            self.model.add(layers.Dense(self.output_dim, activation = None, input_shape = (self.input_dim,)))

        else:
            for i, layer in enumerate(self.arquitectura):
                #print('NUEVA CAPA', i, layer)
                if i == 0:
                    self.model.add(layers.Dense(layer[1], activation = layer[0], input_shape = (self.input_dim,), kernel_initializer = initializador)) # Capa de entrada
                else:
                    self.model.add(layers.Dense(layer[1], activation = layer[0], kernel_initializer = initializador)) # Cualquier otra capa
            
        # Agrega la capa de salida, con y_test.shape[1] neuronas y sin activación (para que quede libre, y no restringir el número a un no negativo (por ejemplo)
        self.model.add(layers.Dense(1, activation = dic_metricas[self.metrica])) # Rend puede ser negativo, Var no
        
        # Compliación y definición
        self.model.compile(loss = loss, optimizer = optimizer, metrics = metrics)
        
        #print('OK estructura FFNN creada')
        
        """
        for i in range(len(self.arquitectura) + 1):
            weights0_origen, biases0_origen = self.model.layers[i].get_weights()
            print('Capa estr', i)
            print('W estr', weights0_origen.shape)
            print('b estr', biases0_origen.shape)
            print(weights0_origen)
            print(biases0_origen)
        """

        #print('\n\n\n')
        return None 
    
    # cambiar_ffnn(self, cambio, capa_seleccionada, delta_n_neurs_new)
    def cambiar_ffnn(self):
        # Aqui los cambios
        #print('CAMBIAR FFNN')
        #print(self.cambio)
        
        #print('NAME REV3')
        #print(self.str_arquitectura, self.arquitectura)
        
        # 1.3 Agregar neuronas
        if self.cambio == '1_agregar_neuronas':
            
            #print('CREAR', self.arquitectura)
            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada']
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            #### 1.3.1 matriz anterior
            weights0_origen, biases0_origen = self.model_base.layers[capa_seleccionada].get_weights()
            #weights0_origen.shape, biases0_origen.shape

            # a weights0_origen, se le hace un hstack random
            filas_add, cols_add = weights0_origen.shape[0], delta_n_neurs_new
            A = np.random.rand(filas_add, cols_add)
            weights0_nuevo = np.hstack([weights0_origen, A]) # Se añaden las columnas nuevas

            # a biases0_origen, se le hace un hstack random
            A = np.random.rand(cols_add)
            biases0_nuevo = np.hstack([biases0_origen, A])

            self.model.layers[capa_seleccionada].set_weights([weights0_nuevo, biases0_nuevo]) # Los setea en la red neuronal"""

            #### 1.3.2 matriz siguiente
            weights1_origen, biases1_origen = self.model_base.layers[capa_seleccionada + 1].get_weights()
            weights1_origen.shape, biases1_origen.shape

            # a weights0_origen, se le hace un vstack random
            filas_add, cols_add = delta_n_neurs_new, weights1_origen.shape[1]
            A = np.random.rand(filas_add, cols_add)
            weights1_nuevo = np.vstack([weights1_origen, A]) # Se añaden las columnas nuevas

            self.model.layers[capa_seleccionada + 1].set_weights([weights1_nuevo, biases1_origen]) # Los setea en la red neuronal"""
            
            #print('Nueva estructura ajustada')
        
        elif self.cambio == '2_eliminar_neuronas':
                        
            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada'] 
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            dic_betas = self.dic_cambios['dic_betas']

            lista_betas_seleccion = dic_betas[capa_seleccionada] # errores imputados a las neuronas en las capas
            
            #print('dic betas', dic_betas)
            df_betas_seleccion = pd.DataFrame(lista_betas_seleccion, columns = ['BETA']) # Se seleccionan las neuronas que serán eliminadas
            df_betas_seleccion['NEURONA'] = df_betas_seleccion.index
            df_betas_seleccion['BETA_ABS'] = np.abs(df_betas_seleccion['BETA'])
            df_betas_seleccion = df_betas_seleccion.sort_values('BETA_ABS', ascending = False).reset_index(drop = True) # Se eligen las neuronas con betas más altos en valr abs
            n_neurs_a_eliminar = abs(self.dic_cambios['delta_n_neurs_new'])

            df_betas_seleccion = df_betas_seleccion.head(n_neurs_a_eliminar)
            neurs_a_eliminar = list(df_betas_seleccion['NEURONA'].unique())
            neurs_a_eliminar.sort()

            lista_representativa = [] # Lista representativa: 0 si es una neurona a eliminar y 1 si es una neurona a conservar
            for i in range(len(lista_betas_seleccion)):
                if i in neurs_a_eliminar:
                    lista_representativa.append(0)
                else:
                    lista_representativa.append(1)

            Ty = generar_matriz_transformacion(lista_representativa, eliminar = 'columnas') # matriz de transformación para eliminar columnas
            Tx = generar_matriz_transformacion(lista_representativa, eliminar = 'filas') 

            #### 1.3.1 matriz anterior (eliminar columnas)
            weights0_origen, biases0_origen = self.model_base.layers[capa_seleccionada].get_weights()  # cambiar por self.model.layers[capa_seleccionada].get_weights()

            weights0_nuevo = weights0_origen @ Ty # cambios en W (se eliminan cols, transformando por derecha)
            biases0_nuevo = Tx @ biases0_origen # cambios en b (se eliminan filas, transformando por izquierda)
            self.model.layers[capa_seleccionada].set_weights([weights0_nuevo, biases0_nuevo]) # seteo de las nuevas configuraciones

            #### 1.3.2 matriz siguiente
            weights1_origen, biases1_origen = self.model_base.layers[capa_seleccionada + 1].get_weights()

            weights1_nuevo = Tx @ weights1_origen # cambios en W (se eliminan filas, transformando por izquierda)
            # biases1_origen se mantiene
            self.model.layers[capa_seleccionada + 1].set_weights([weights1_nuevo, biases1_origen]) # seteo de las nuevas configuraciones
            
        elif self.cambio == '3_eliminar_capa': # ver hojas 16 & 17

            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada'] # cambiar por self.dic_cambios
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            
            for i in range(len(self.arquitectura) + 2): # +1 por capa de salida + 1 por capa de arquitectura original
                weights0_origen, biases0_origen = self.model_base.layers[i].get_weights()
                if i < capa_seleccionada:
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se mantiene
                elif (i == capa_seleccionada) or (i == capa_seleccionada + 1): # se conserva el random generado (se pierde la matriz antecesora y sucesora de la capa)
                    None
                else:
                    self.model.layers[i - 1].set_weights([weights0_origen, biases0_origen]) # se mantiene
        
        elif self.cambio == '3_agregar_capa': # Ver hojas 17 & 18
            self.crear_ffnn()
            #{'capa_seleccionada': 3, 'neuronas_seleccionadas': 4, 'funcion_seleccionada': 'relu'}
            capa_seleccionada = self.dic_cambios['capa_seleccionada']
            for i in range(len(self.arquitectura) + 1): # +1 por capa de salida 
                if i < capa_seleccionada:
                    weights0_origen, biases0_origen = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se reemplazan en la nueva red directamente
                elif (i == capa_seleccionada) or (i == capa_seleccionada + 1): # se conserva el random generado (se pierde la matriz antecesora y sucesora de la capa)
                    None
                else:
                    weights0_origen, biases0_origen = self.model_base.layers[i - 1].get_weights()
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen])
        
        elif (self.cambio == '4_cambio_funcion') or (self.cambio == '5_cambio_funcion'):
            
            self.crear_ffnn() # Se crea la nueva estructura con la nueva función de activación
            
            for i in range(len(self.arquitectura) + 1): # +1 por capa de salida 
                weights0_origen, biases0_origen = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se reemplazan en la nueva red directamente
        
        elif self.cambio == '6_cambios_inputs_agregar':
            
            # AGREGAR!!!
            self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
            self.crear_ffnn() # Se crea la nueva estructura
                        
            lista_representativa = [] # Lista representativa: 0 si es una neurona nueva (se generará un random en la función generar_matriz_transformacion_estructurada) y 1 si es una neurona a conservar
            for elemento in self.campos_input:
                if elemento in self.campos_input_heredados:
                    lista_representativa.append(1)
                else:
                    lista_representativa.append(0)

            Tx = generar_matriz_transformacion_estructurada(lista_representativa)
            
            # Cambio en las matrices de input
            for i in range(len(self.arquitectura) + 1):
                weights, biases = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                if i == 0:
                    weights = Tx @ weights # cambios en W (se eliminan filas, transformando por izquierda)
                self.model.layers[i].set_weights([weights, biases]) # se reemplazan en la nueva red directamente
            
                
        elif self.cambio == '7_cambios_inputs_eliminar':
            
            self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
            self.crear_ffnn() # Se crea la nueva estructura
            
            lista_representativa = [] # Lista representativa: 0 si es ya no existe y 1 si es una neurona a conservar
            for elemento in self.campos_input_heredados:
                if elemento in self.campos_input:
                    lista_representativa.append(1)
                else:
                    lista_representativa.append(0)
            
            Tx = generar_matriz_transformacion(lista_representativa, 'filas')
            
            # Cambio en las matrices de input
            for i in range(len(self.arquitectura) + 1):
                weights, biases = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                if i == 0:
                    weights = Tx @ weights # cambios en W (se eliminan filas, transformando por izquierda)
                self.model.layers[i].set_weights([weights, biases]) # se reemplazan en la nueva red directamente

        return None
                
    def inicializar(self, campos_input, campos_output): # Si la red no fue rescatada, entonces se leen algunos parámetros
        #print('Inicializar')
        
        #print('campos_input', campos_input)
        # Estandarización de campos input y output
        if campos_input == None:
            self.campos_inputs_default() # Se generan los campos  inputs por  default: Todos los existentes en los valores que están en df_activos
        elif type(campos_input) == str:
            self.campos_input = campos_input.split(',')
        else:
            self.campos_input = campos_input
        self.campos_input.sort()
        
        self.campos_output = campos_output.split(',')
        self.campos_output.sort()
        
        self.str_campos_input, self.str_campos_output = ','.join(self.campos_input),  ','.join(self.campos_output)
        self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
        
        #print(self.campos_input, self.campos_output)
        #print('dims', self.input_dim, self.output_dim)
        
        # Reconoce si existe, o crea un id nuevo
        if 'df_info_ffnn.pkl' not in os.listdir(self.seguimiento):
            self.df_info_ffnn = pd.DataFrame(columns = ['str_arquitectura', 'campos_input', 'campos_output', 'nombre'])
            #return None
        else:
            self.df_info_ffnn = pickle_act(f'{self.seguimiento}df_info_ffnn') # Se lee si existe
            
        # Asignación de nombre...se rescata si existe, de lo contrario se asigna uno nuevo
        df_info_ffnn_filtrado = self.df_info_ffnn[(self.df_info_ffnn['str_arquitectura'] == self.str_arquitectura) & (self.df_info_ffnn['campos_input'] == self.str_campos_input) & (self.df_info_ffnn['campos_output'] == self.str_campos_output)].reset_index(drop = True) 
        
        if len(df_info_ffnn_filtrado) == 0: # asignación de nombre
            self.nombre = 'FFNN_'+ str(len(self.df_info_ffnn) + 1) # Se asigna un nombre con un id
            # Actualización de df info y guardado
            new_row = pd.DataFrame({'str_arquitectura': [self.str_arquitectura], 'campos_input': [self.str_campos_input], 'campos_output': [self.str_campos_output], 'nombre': [self.nombre]})
            self.df_info_ffnn = pd.concat([self.df_info_ffnn, new_row], axis = 0).reset_index(drop = True)
            pickle_act(f'{self.seguimiento}df_info_ffnn', variable = self.df_info_ffnn, mode = 'save') # Se guarda, solo si se agrega un caso nuevo
        else:
            self.nombre = df_info_ffnn_filtrado['nombre'][0]

        return None
    
    
    def campos_inputs_default(self):
        
        #print('campos_inputs_default')
        cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
        set_campos_inputs = set()
        for i in range(len(self.df_activos)):
            simbolo, nombre = self.df_activos.loc[i]
            # Si no existe el objeto, se crea
            valor = Valor(simbolo, nombre, cofre0) # 
            if len(valor.raw_x) == 0: # No hay datos que aportar
                continue
            set_campos_inputs_new = set([campo for campo in valor.raw_x['NAME'].unique() if campo[:2] != "Y_"])
            #print('set_campos_inputs_new')
            #print(set_campos_inputs_new)
            set_campos_inputs = set_campos_inputs.union(set_campos_inputs_new)
        self.campos_input = list(set_campos_inputs)
        
        #self.campos_input = self.campos_input[:10] # fijado: cambiar!!
        
        return None
    
    def construir_matrices(self):
        
        while True:
        
            # Matrices de input y de output
            #print('Campos inputs en construir matrices')
            #print(self.campos_input)
            #print(self.campos_output)
            # output_level = 10 # 10 días para este ejemplo (esta red neuronal concretamente)
            cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
            df_raw_info = pd.DataFrame()
            for i in range(len(self.df_activos)):
                simbolo, nombre = self.df_activos.loc[i]
                valor = Valor(simbolo, nombre, cofre0) # Si no existe el objeto, se crea 
                if len(valor.raw_x) == 0: # No hay datos que aportar
                    continue
                #print(simbolo, nombre, len(valor.raw_x))
                new_raw_x = valor.raw_x.copy()
                new_raw_x['VALOR'] = simbolo
                
                print('SIMBOLO', simbolo, new_raw_x['DATE'].min(), new_raw_x['DATE'].max())
                df_raw_info = pd.concat([df_raw_info, new_raw_x], axis = 0)
            

            #display(df_raw_info[df_raw_info['X'].isna()])
            #sys.exit('df raw info nan')
            df_raw_info = df_raw_info[['VALOR', 'DATE', 'NAME', 'X']]
            #df_raw_info.head()
            
            #print('Name en raw info')
            #print(df_raw_info['NAME'].unique())
            df_raw_info = df_raw_info[(df_raw_info['NAME'].isin(self.campos_input)) | (df_raw_info['NAME'].str[:2] == 'Y_')] # Se seleccionan solo los campos de input que están declarados en campos input
            #sys.exit('Revisar esto (deberian seleccionarse solo los campos inputs)!')

            # Primero, se separan los inputs de los outputs
            df_raw_info['NATURALEZA'] = np.where(df_raw_info['NAME'].str[:2] == 'Y_', 'Y', 'X')
            df_raw_y = df_raw_info[df_raw_info['NATURALEZA'] == 'Y'].reset_index(drop = True)
            df_raw_x = df_raw_info[df_raw_info['NATURALEZA'] == 'X'].reset_index(drop = True)

            df_X = df_raw_x.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()
            df_Y = df_raw_y.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()

            df = df_X.merge(df_Y, on = ['VALOR', 'DATE'], how = 'outer')
        
        #df.to_csv(f'dfX_{self.nombre}.csv', index = False, decimal = ',', sep = ';')
        #sys.exit('dfX.csv')
        
            aprobado = True
            for c in list(set(df.columns) - {'DATE', 'VALOR'}):
                if len(df[df[c].isna()]) > 0:
                    aprobado = False
                    print(f'El campo {c} tiene valores nulos')
                    display(df[df[c].isna()])
                    sys.exit('Salida por campos nulos')
            if aprobado:
                break
            print('B. Espera de completitud de valores en Valor.py: Espera de 30s.')
            time.sleep(30)
        
        """
        for c in list(set(df.columns) - {'DATE', 'VALOR'}):
            if len(df[df[c].isna()]) > 0:
                print(c)
                display(df[df[c].isna()])
                sys.exit('Error en campo')
        """

        # Limpieza de df
        self.lista_campos_output = list(set(df_Y.columns) - {'DATE', 'VALOR'})
        set_delta_dates = set()
        for c in self.lista_campos_output:

            delta = c.split('_')[-1]
            set_delta_dates.add(delta)
        #lista_delta_dates = list(set_delta_dates)
        
        df['PREDICT'] = np.where((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0), True, False)
        #print('lista_delta_dates', lista_delta_dates)
        df, self.df_normalizacion = normalizar(df)

        #df.to_csv(f'dfX_v2_{self.nombre}.csv', index = False, decimal = ',', sep = ';')
        
        #print('La idea es normalizar esta BD')
        #sys.exit(f'Salida para revisión de dfX_v2_{self.nombre}.csv')
        
        delta = self.output_level

        print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')            
        self.df_predict = df[df['PREDICT']].reset_index(drop = True) # Se aislan los datos para después hacer el predict
        self.df_predict = self.df_predict.drop(columns = ['PREDICT'])
        if len(self.df_predict) == 0:
            sys.exit('Predict sin datos')
        else:
            None
            #print(self.nombre)
            #print(self.df_predict)
        df = df[~df['PREDICT']].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
        df = df.drop(columns = ['PREDICT'])
    
        
        """
        for delta in lista_delta_dates:
            print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')            
            self.df_predict = df[((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se aislan los datos para después hacer el predict
            if len(self.df_predict) == 0:
                sys.exit('Predict sin datos')
            else:
                print(self.nombre)
                print(self.df_predict)
            df = df[~((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
        """
        
        if self.output_level not in self.dic_data:
            self.dic_data[self.output_level] = df
        
        return None
    
    def split_data(self, test_size = 0.2): # Como método propio, para poder generar sampleos aleatorios libremente
        
        df = self.dic_data[self.output_level].copy()
        df = df.fillna(0) # Provisorio, para activos que están incompletos con sus datos históricos
        df = df.drop(columns = ['DATE', 'VALOR'])
        Y = df[f'Y_{self.metrica}_{self.output_level}'] # Una sola métrica de output ya que si es varianza,hay que asegurar que sea >= 0
        X = df.drop(columns = self.lista_campos_output)
        x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = test_size) # Split aleatorio de los datos
        x_train, x_test, y_train, y_test = x_train.values, x_test.values, y_train.values, y_test.values # dt to matriz numpy
        return x_train, x_test, y_train, y_test
    
    def matrices_predict(self): # nuevo 240614
        df_x_predict = self.df_predict.drop(columns = self.lista_campos_output) 
        df_x_predict_values = df_x_predict.drop(columns = ['DATE', 'VALOR'])
        x_predict = df_x_predict_values.values
        return df_x_predict, x_predict
    
    def predict_model(self, x_predict):
        y_predict = self.model.predict(x_predict)
        return y_predict
    
    def train_model(self, x_train, x_test, y_train, y_test, batch_size = 32, epochs = 5, n_min = 200, plotear = False): # n_min: cuantos epochs sin superar el best son necesarios para stop
        
        #print('EN TRAIN MODEL\n\n\n\n\n')
        if not plotear:
            e, k = 0, 0
            while True:
                #print('K', k)

                self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), callbacks = [Callback()], verbose = 0) # Entrenamiento del modelo
                # batch_size: cuantos datos juntos se entrenan a la vez, antes de actualizar los parámetros
                # verbose = 0: muestra menos información de output
                # self.train_loss, self.train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
                #print('W-b en train')
                [w, b] = self.model.layers[0].get_weights()
                #print(w)
                #print(b)
                self.test_loss, self.test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
                #print(\test_loss, test_accuracy\, self.test_loss, self.test_accuracy)
                if self.test_loss < self.best_test_loss: # Se guardan los mejores registros y el mejor modelo para la predicción (si es que los parámetros mejoran)
                    self.best_test_loss = self.test_loss
                    self.best_test_accuracy = self.test_accuracy
                    self.best_model = self.model
                    k = 0 # reset
                k += 1 # contador de cuantos epochs van sin mejorar el best loss
                if k >= n_min: # criterio de salida
                    break
                
            return None
        
        # plotear = True
        df_plot = pd.DataFrame()
        for e in range(epochs):
            print('epoch', e)
            self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), verbose = 0)
            
            [w, b] = self.model.layers[0].get_weights()
            #print('W-b en train')
            #print(w)
            #print(b)
            train_loss, train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
            test_loss, test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
            new_df = pd.DataFrame({'EPOCH': [e], 'TRAIN_LOSS': [train_loss], 'TRAIN_ACCURACY': [train_accuracy], 'TEST_LOSS': [test_loss], 'TEST_ACCURACY': [test_accuracy]})
            display(new_df)
            df_plot = pd.concat([df_plot, new_df], axis = 0)
        
        # plotea
        fig, ax = plt.subplots(1, 2, figsize = (15, 5))
        ax[0].plot(df_plot['EPOCH'], df_plot['TRAIN_LOSS'], label = 'TRAIN LOSS')
        ax[0].plot(df_plot['EPOCH'], df_plot['TEST_LOSS'], label = 'TEST LOSS')
        ax[0].legend()
        ax[0].set_title('LOSS')
        
        ax[1].plot(df_plot['EPOCH'], df_plot['TRAIN_ACCURACY'], label = 'TRAIN ACCURACY')
        ax[1].plot(df_plot['EPOCH'], df_plot['TEST_ACCURACY'], label = 'TEST ACCURACY')
        ax[1].legend()
        ax[1].set_title('ACCURACY')
        plt.show()
        
        self.df_plot = df_plot.copy()
        
        return None
        
        # plotear = True
        df_plot = pd.DataFrame()
        for e in range(epochs):
            self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), verbose = 0)
            train_loss, train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
            test_loss, test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
            new_df = pd.DataFrame({'EPOCH': [e], 'TRAIN_LOSS': [train_loss], 'TRAIN_ACCURACY': [train_accuracy], 'TEST_LOSS': [test_loss], 'TEST_ACCURACY': [test_accuracy]})
            df_plot = pd.concat([df_plot, new_df], axis = 0)
        
        # plotea
        fig, ax = plt.subplots(1, 2, figsize = (15, 5))
        ax[0].plot(df_plot['EPOCH'], df_plot['TRAIN_LOSS'], label = 'TRAIN LOSS')
        ax[0].plot(df_plot['EPOCH'], df_plot['TEST_LOSS'], label = 'TEST LOSS')
        ax[0].legend()
        ax[0].set_title('LOSS')
        
        ax[1].plot(df_plot['EPOCH'], df_plot['TRAIN_ACCURACY'], label = 'TRAIN ACCURACY')
        ax[1].plot(df_plot['EPOCH'], df_plot['TEST_ACCURACY'], label = 'TEST ACCURACY')
        ax[1].legend()
        ax[1].set_title('ACCURACY')
        plt.show()
        
        self.df_plot = df_plot.copy()
        
        return None
    
    def error_imputado_neuronas(self, x_train, y_train):
        dic_metricas = {'Rendimiento': None, 'Varianza': 'relu'}
        dic_funciones = {'sigmoid': lambda x: 1 / (1 + np.exp(-x)), None: lambda x: x, 'relu': lambda x: np.maximum(0, x), 'tanh': lambda x: np.tanh(x)}
        dic_derivadas = {'sigmoid': lambda x: x * (1 - x), None: lambda x: 1, 'relu': lambda x: 1 if x > 0 else 0, 'tanh': lambda x: 1 - x ** 2}

        # Algoritmo de obtención de datos
        
        #print(self.model.layers)
        #print(self.arquitectura)
        #for i in range(len(self.model.layers)):
        #    [w, b] = self.model.layers[i].get_weights()
        #    print(i, w.shape, b.shape)
        #sys.exit('Revisar 240524')

        dic_ecuaciones = {}

        # 0. Inicio
        x = x_train
        
        for capa, detalle in enumerate(self.arquitectura):
            [act_fun, n_neurs] = detalle
            #print(capa, act_fun, n_neurs)
            # self.model.layers[capa].get_weights()
            [w, b] = self.model.layers[capa].get_weights()
            if np.isnan(w[0][0]):
                sys.exit('W vacío')
            z = x @ w + b 
            a = dic_funciones[act_fun](z)
                        
            dic_ecuaciones[capa] = {'z': z, 'a': a, 'x': x, 'w': w, 'b': b, 'act_fun': act_fun}
            x = a
            

        # Capa salida
        capa, act_fun, n_neurs = capa + 1, dic_metricas[self.metrica], 1
        #print(capa, act_fun, n_neurs)
        [w, b] = self.model.layers[capa].get_weights()
        z = x @ w + b 
        a = dic_funciones[act_fun](z)
        dic_ecuaciones[capa] = {'z': z, 'a': a, 'x': x, 'w': w, 'b': b, 'act_fun': act_fun}

        # 1. Obtener responsabilidad de cada neurona
        dic_deltas = {}
        # 1.1 Para la ultima capa
        # dL = (a - y) * f'(z)

        a = dic_ecuaciones[capa]['a']
        z = dic_ecuaciones[capa]['z']
        y = y_train.reshape(-1, 1)
        
        derivada = np.vectorize(dic_derivadas[act_fun])(z)
        dL = (a - y) * derivada
        dic_deltas[capa] = dL

        # Para cualquier capa intermedia
        # dL = (dL @ w.T) * f'(z)

        while True:
            w = dic_ecuaciones[capa]['w']
            if capa > 0:
                act_fun_prev = dic_ecuaciones[capa - 1]['act_fun']
                z_prev = dic_ecuaciones[capa - 1]['z']
                a_prev = dic_ecuaciones[capa - 1]['a']
            dL_next = dic_deltas[capa]

            if capa == 0:
                derivada_prev = 1
            elif act_fun_prev == 'relu': # La derivada se aplica a x, y no a f(x)
                derivada_prev = np.vectorize(dic_derivadas[act_fun_prev])(z_prev)
            else:
                derivada_prev = np.vectorize(dic_derivadas[act_fun_prev])(a_prev) # ya que a = f(z) y las derivadas de sigm y tanh están definidas en función de f(z)

            dL = (dL_next @ w.T) * derivada_prev
            capa -= 1
            dic_deltas[capa] = dL
            
            #print('Capa y dL', capa, dL)
            
            if capa == -1: # capa 0 es la primera capa oculta
            #if capa == 0: # capa 0 es la primera capa oculta
                break

        # betas
        self.dic_betas = {}
        for capa in dic_deltas:
            beta = dic_deltas[capa].sum(axis = 0)
            self.dic_betas[capa] = beta
            #print(capa, beta)
        
        return None
    
    #print('C:\Users\mvaldiviad\OneDrive - Falabella\Escritorio\Proyectos Personales\Markowitz\0. Markowitz FFNN 240322 V1_traspaso a ALginvesting.ipynb')
    

# Clase Búsqueda Inteligente

In [ ]:
def agregar_o_eliminar_inputs(campos_inputs, all_campos_inputs, modo, beta_inputs = []):
    print('campos_inputs', len(campos_inputs), modo)
    print(campos_inputs)
    
    dic_cambios = {}
    if modo == 'agregar':
        delta_opciones = all_campos_inputs - set(campos_inputs)
        delta_agr = len(delta_opciones)
        if delta_agr == 0:
            print('No se pueden agregar más campos')
            dic_cambios['continuar'] = False

        else:
            # elegir random entre 1 y delta_agr
            delta_agr_new = np.random.choice(range(1, delta_agr + 1), p = obtener_lista_probs(delta_agr))
            # elegir delta_agr_new campos random de delta_opciones
            campos_inputs_new = list(np.random.choice(list(delta_opciones), delta_agr_new, replace = False))
            
            print('\n\n\ndelta new', delta_agr_new)
            dic_cambios['continuar'] = True
            campos_inputs += campos_inputs_new
            dic_cambios['campos_inputs_new'] = campos_inputs
            
    else: # modo "eliminar"
        delta_eliminar = len(campos_inputs) - 1 # tiene que quedar al menos un campo de input
        if delta_eliminar == 0:
            print('No se pueden eliminar más campos')
            dic_cambios['continuar'] = False
        
        else:
            delta_eliminar_new = np.random.choice(range(1, delta_eliminar + 1), p = obtener_lista_probs(delta_eliminar))
            #campos_a_eliminar = list(np.random.choice(campos_inputs, delta_eliminar_new, replace = False))
            
            print('\n\n\ndelta_eliminar_new', delta_eliminar_new)
            if len(beta_inputs) == 0: # caso random, en caso de que no exista entrenamiento
                # crea una lista random uniforme (0,1) de tamaño len(campos_inputs)
                beta_inputs = np.random.rand(len(campos_inputs))

            #print('beta_inputs', beta_inputs)
            df_betas_seleccion = pd.DataFrame(beta_inputs, columns = ['BETA']) # Se seleccionan las neuronas que serán eliminadas
            df_betas_seleccion['IDX'] = df_betas_seleccion.index
            
            df_neurs = pd.DataFrame(campos_inputs, columns = ['NEURONA_INPUT'])
            df_neurs['IDX'] = df_neurs.index
            
            df_betas_seleccion = df_betas_seleccion.merge(df_neurs, on = 'IDX', how = 'left')
        
            df_betas_seleccion['BETA_ABS'] = np.abs(df_betas_seleccion['BETA'])
            df_betas_seleccion = df_betas_seleccion.sort_values('BETA_ABS', ascending = False).reset_index(drop = True) # Se eligen las neuronas con betas más altos en valr abs

            df_betas_seleccion = df_betas_seleccion.head(delta_eliminar_new)
            neurs_a_eliminar = list(df_betas_seleccion['NEURONA_INPUT'].unique())
            neurs_a_eliminar.sort()
            
            #display(df_betas_seleccion)
            
            for n in neurs_a_eliminar:
                campos_inputs.remove(n)
                
            dic_cambios['continuar'] = True
            dic_cambios['campos_inputs_new'] = campos_inputs # Solo se entrega delta_eliminar_new, ya que los inputs eliminados se eligen según el criterio de los betas
    
    return dic_cambios
            


In [ ]:
def agregar_o_eliminar_neuronas(lista_n_neurs, str_arquitectura, modo):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura
    # 1. Agregar neuronas

    dic_n_neurs = {n_neurs: i for i, n_neurs in enumerate(lista_n_neurs)}
    dic_n_neurs_inv = {i: n_neurs for i, n_neurs in enumerate(lista_n_neurs)}

    # 1.1 Se revisa que capas son potenciales para agregar neuronas
    dic_neurs_potenciales, dic_n_new = {}, {}
    max_n_neurs = max(lista_n_neurs)
    for capa, detalle in enumerate(arquitectura):
        [act_fun, n_neurs] = detalle
        #print(capa, act_fun, n_neurs)
        if n_neurs == max_n_neurs: # No se pueden agregar mas neuronas en esta capa
            continue
        id_posicion = dic_n_neurs[n_neurs]
        id_posicion_new = id_posicion + 1 # para agregar
        if modo == 'eliminar':
            id_posicion_new = id_posicion - 1 # para eliminar
        n_neurs_new = dic_n_neurs_inv[id_posicion_new] # cual es el n_neurs que viene en lista_n_neurs
        delta_neurs = n_neurs_new - n_neurs # cuantas neuronas se agregarían
        dic_neurs_potenciales[capa] = delta_neurs
        dic_n_new[capa] = n_neurs_new

    #print('dic_neurs_potenciales')
    #print(dic_neurs_potenciales)
    
    # 1.2 Elegir random una de las capas potenciales para agregar neuronas
    capa_opciones = list(dic_neurs_potenciales.keys())

    # elige uno al azar y equiprobable con random choice
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada
    delta_n_neurs_new = dic_neurs_potenciales[capa_seleccionada] # y cuantas neuronas se agregarán
    n_news = dic_n_new[capa_seleccionada] # cuantas neuronas en total tendrá la capa seleccionada

    # Nombre de la nueva arquitectura
    #print('Ante cualquier modificación definida, revisar antes que todo, si esa estructura ya existe y está guardada')
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = str_arquitectura_new[capa_seleccionada].split('_')
    str_arquitectura_new_capa = '_'.join([str_arquitectura_new_capa[0], str(int(str_arquitectura_new_capa[1]) + delta_n_neurs_new)])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada + 1:]
    str_arquitectura_new = ','.join(str_arquitectura_new)
    #print(str_arquitectura_new)

    dic_cambios = {'capa_seleccionada': capa_seleccionada, 'delta_n_neurs_new': delta_n_neurs_new, 'n_news': n_news}
    
    return str_arquitectura_new, dic_cambios

def agregar_nueva_capa(lista_n_neurs, str_arquitectura):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura

    lista_n_neurs_new_layer = lista_n_neurs.copy()
    lista_n_neurs_new_layer.remove(0)

    # identificar etapa de crecimiento y decrecimiento

    n_capas = [arquitectura[i][1] for i in range(len(arquitectura))]
    dic_cambios_fase = {}
    for i in range(1, len(n_capas)):
        if n_capas[i] < n_capas[i - 1]:
            dic_cambios_fase['decrecimiento'] = i
            break

    for i in range(len(n_capas) - 2, -1, -1):
        if n_capas[i] < n_capas[i + 1]:
            dic_cambios_fase['crecimiento'] = i
            break

    dic_etapas = {}
    for i in range(len(n_capas)):
        if ('crecimiento' in dic_cambios_fase) and (i <= dic_cambios_fase['crecimiento']):
            dic_etapas[i] = 'crecimiento'
        elif ('decrecimiento' in dic_cambios_fase) and (i >= dic_cambios_fase['decrecimiento']):
            dic_etapas[i] = 'decrecimiento'
        else:
            dic_etapas[i] = 'estable'

    # COMBINACIONES: c-c (entre), c-e (>= n neurs capa c), c-d (no existe), e-e (>= n neurs capa e (cualquiera, las dos tienen lo mismo)), e-d (>= n neurs capa d), d-d (entre)
    # Puedo validar la arquitectura post...si existe un cambio != a los de arriba, está mal

    dic_neurs_potenciales = {}
    for j in range(len(arquitectura) + 1):
        if j == 0: # inicial (antes de la capa 0)
            lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[0]] # casos <= n_neurs de capa inicial
        elif (j > 0) and (j != len(arquitectura)):
            sigla_cambio = dic_etapas[j - 1][0] + '-' + dic_etapas[j][0]
            if sigla_cambio == 'c-c':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j - 1] and k <= n_capas[j]]
            if sigla_cambio == 'c-e':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j - 1]]
            if sigla_cambio == 'c-d':
                sys.exit('Esta combinación no debería existir')
            if sigla_cambio == 'e-e':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j]]
            if sigla_cambio == 'e-d':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j]]
            if sigla_cambio == 'd-d':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[j - 1] and k >= n_capas[j]]
        else:
            lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[j - 1]]
        
        dic_neurs_potenciales[j] = lista_opciones

    # Elegir que capa se agregará
    capa_opciones = list(dic_neurs_potenciales.keys())
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada

    neuronas_potenciales = dic_neurs_potenciales[capa_seleccionada]
    neuronas_seleccionadas = np.random.choice(neuronas_potenciales) # n neurs seleccionadas para la capa

    lista_funciones = ['sigmoid', 'relu', 'tanh']
    funcion_seleccionada = np.random.choice(lista_funciones)

    ###
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = '_'.join([funcion_seleccionada, str(neuronas_seleccionadas)])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada:]
    str_arquitectura_new = ','.join(str_arquitectura_new)

    #print(str_arquitectura_new)
    dic_cambios = {'capa_seleccionada': capa_seleccionada, 'neuronas_seleccionadas': neuronas_seleccionadas, 'funcion_seleccionada': funcion_seleccionada}

    return str_arquitectura_new, dic_cambios

def cambiar_funcion(str_arquitectura, modo):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura
    
    # 1.2 Elegir random una de las capas potenciales para agregar neuronas
    capa_opciones = list(range(len(arquitectura)))

    # elige uno al azar y equiprobable con random choice
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada
    
    funcion_seleccionada = arquitectura[capa_seleccionada][0]
    
    cambios_funcion = {'siguiente': {'tanh': 'relu', 'relu': 'sigmoid', 'sigmoid': 'tanh'},
                       'anterior': {'relu': 'tanh', 'sigmoid': 'relu', 'tanh': 'sigmoid'}}
    
    nueva_funcion = cambios_funcion[modo][funcion_seleccionada]
    
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = str_arquitectura_new[capa_seleccionada].split('_')
    str_arquitectura_new_capa = '_'.join([nueva_funcion, str_arquitectura_new_capa[1]])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada + 1:]
    str_arquitectura_new = ','.join(str_arquitectura_new)
    
    dic_cambios = {'nueva_funcion': nueva_funcion, 'modo': modo}

    return str_arquitectura_new, dic_cambios

def ajustar_str_arquitectura_eliminar_capa(str_arquitectura_new):
    capas = str_arquitectura_new.split(',')
    capas_new = []
    for capa in capas:
        if capa[-2:] == '_0':
            continue
        capas_new.append(capa)
    capas_new
    str_arquitectura_new = ','.join(capas_new)
    return str_arquitectura_new


In [ ]:
def Probabilidad_Poisson(lambd, k):
    return (lambd ** k) * np.exp(-lambd) / np.math.factorial(k)


def obtener_lista_probs(n_max, factor = 0.3): # Factor -> lambda = factor * max_n
    df_probs = pd.DataFrame({'k': list(range(1, n_max + 1))})
    max_k = df_probs['k'].max()
    df_probs['Pr'] = df_probs['k'].apply(lambda x: Probabilidad_Poisson(max_k * factor, x))
    S = df_probs['Pr'].sum()
    df_probs['Pr'] = df_probs['Pr'] / S
    list_probs = list(df_probs['Pr'])
    return list_probs

In [ ]:
class Busqueda_Inteligente():
    
    def __init__(self, nombre, metrica, output_level, cofre, lista_n_neurs, batch_size = 32, epochs = 50):
        
        n_movs = 7
        
        self.nombre = f'{nombre}_{metrica}_{output_level}'
        self.metrica = metrica
        self.output_level = output_level
        self.batch_size = batch_size
        self.epochs = epochs
        self.cofre = f'{cofre}Busqueda_Inteligente/'
        self.lista_n_neurs = lista_n_neurs
        self.cofre_base = cofre
        self.campos_input = None
        self.str_arquitectura = "tanh_4,relu_8,sigmoid_4" # arquitectura inicial (si no existe nada)
        self.inicial = True
        
        #print('df activos 0 en busqint')
        #display(df_activos)
        self.df_activos = df_activos
        self.best_test_loss_global = float('inf')
        self.dic_movimientos = {i: 0 for i in range(1, n_movs + 1)}
        self.obtener_all_campos_inputs() # obtiene cuantos campos inputs diferentes hay en total
        self.rescatar() # Si existe, se rescatan los atributos de la clase

        #print('df activos 1 en busqint')
        #display(self.df_activos)
        
        return None
    
    def rescatar(self):
        
        print(f'{self.nombre}.pkl')
        print(os.listdir(self.cofre))
        
        if f'{self.nombre}.pkl' not in os.listdir(self.cofre):
            return None # Si no se encuentra, no se pueden rescatar los atributos
    
        print(f'Rescate en {self.cofre}') # eliminar
        valor_cargado = pickle_act(f'{self.cofre}{self.nombre}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos
        for key, value in vars(valor_cargado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
            if key in ['df_activos']: # atributos no heredados
                continue
            setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
        return None
    
    def obtener_all_campos_inputs(self):
            
        print('campos_inputs_default en busq int')
        cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
        all_campos_inputs = set()
        for i in range(len(self.df_activos)):
            simbolo, nombre = self.df_activos.loc[i]
            # Si no existe el objeto, se crea
            valor = Valor(simbolo, nombre, cofre0) # 
            if len(valor.raw_x) == 0: # No hay datos que aportar
                continue
            set_campos_inputs_new = set([campo for campo in valor.raw_x['NAME'].unique() if campo[:2] != "Y_"])
            all_campos_inputs = all_campos_inputs.union(set_campos_inputs_new)
        self.all_campos_inputs = all_campos_inputs
        print('     len', len(all_campos_inputs))
        
        return None
    
    def ffnn_actual(self):
        self.ffnn = Red_Neuronal(self.str_arquitectura, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos)
        self.campos_input = self.ffnn.campos_input
        print(self.str_arquitectura)
        return None
    
    def train(self, n_min):
        self.ffnn.construir_matrices() 
        x_train, x_test, y_train, y_test = self.ffnn.split_data()
        
        self.ffnn.train_model(x_train, x_test, y_train, y_test, batch_size = self.batch_size, n_min = n_min, epochs = None, plotear = False) # plotear mientras!! no se guarda el entrenamiento, solo para validar
        self.best_test_loss_iter = self.ffnn.best_test_loss
        print('BEST TEST LOSS', self.ffnn.best_test_loss)
        self.ffnn.error_imputado_neuronas(x_train, y_train)
        return None

    def elegir_nueva_arquitectura(self, n_min):
        
        max_capas_permitidas = 5
        
        dic_selecciones = {1: 'agregar neuronas', 2: 'eliminar neuronas (o capa)', 3: 'agregar capa', 4: 'cambiar act_function', 5: 'cambiar act_function', 6: 'agregar inputs', 7: 'eliminar inputs'}
        # elegir nueva arquitectura (anclar arriba)
        n_movs = len(self.dic_movimientos)
        
        #dic_movimientos = {i: 0 for i in range(1, n_movs + 1)}

        min_value = min(list(self.dic_movimientos.values()))
        max_value = max(list(self.dic_movimientos.values()))

        if (min_value == 0) and (max_value == 0):
            dic_probs = {i: 1 / len(self.dic_movimientos) for i in range(1, n_movs + 1)}
        else:
            dic_probs = {i: self.dic_movimientos[i] - min_value for i in range(1, n_movs + 1)}
            S = sum(list(dic_probs.values()))
            dic_probs = {i: dic_probs[i] / S for i in range(1, n_movs + 1)}
        
        #print('ELEGIR NUEVA ARQUITECTURA')
        #print('     dic_probs', dic_probs)
        while True:

            # En base a estas probabilidades, elegir una opcion
            x = np.random.choice(list(dic_probs.keys()), p = list(dic_probs.values()))
            #x = 7 # fijado: cambiar!!
            print('\n\n     Nueva seleccion', x, dic_selecciones[x])
            if x == 1:
                # [ok] cambio 1 desarrollado: 1_agregar_neuronas
                str_arquitectura_new, dic_cambios = agregar_o_eliminar_neuronas(self.lista_n_neurs, self.str_arquitectura, 'agregar')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '1_agregar_neuronas', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break

            elif x == 2:
                # [ok] cambio 2: 2_eliminar_neuronas
                str_arquitectura_new, dic_cambios = agregar_o_eliminar_neuronas(self.lista_n_neurs, self.str_arquitectura, 'eliminar')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                
                try:
                    beta_inputs = self.ffnn.dic_betas
                except:
                    self.train(n_min)
                    beta_inputs = self.ffnn.dic_betas
                    
                dic_cambios['dic_betas'] = beta_inputs

                if dic_cambios['n_news'] != 0:
                    self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '2_eliminar_neuronas', dic_cambios = dic_cambios)
                else: # eliminar capa
                    str_arquitectura_new = ajustar_str_arquitectura_eliminar_capa(str_arquitectura_new)
                    self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '3_eliminar_capa', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 3:
                # cambio 3: 3_agregar_capa
                
                if len(self.str_arquitectura.split(',')) == max_capas_permitidas:
                    print(f'No se puede agregar capa porque la arquitectura actual es {self.str_arquitectura} y sse permite un máximo de {max_capas_permitidas} capas')
                    continue

                str_arquitectura_new, dic_cambios = agregar_nueva_capa(self.lista_n_neurs, self.str_arquitectura)
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '3_agregar_capa', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 4:
                # cambio 4: función siguiente
                str_arquitectura_new, dic_cambios = cambiar_funcion(self.str_arquitectura, 'siguiente')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '4_cambio_funcion', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 5:
                # cambio 5: función anterior
                str_arquitectura_new, dic_cambios = cambiar_funcion(self.str_arquitectura, 'anterior')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '4_cambio_funcion', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
            
            elif x == 6:
                         
                str_arquitectura_new = self.str_arquitectura
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.campos_input_heredados = self.campos_input[:]
                dic_cambios = agregar_o_eliminar_inputs(self.campos_input, self.all_campos_inputs, 'agregar')
                if not dic_cambios['continuar']:
                    continue # Elegir un nuevo x, ya que no hay campos que puedan ser agregados
                self.campos_input = dic_cambios['campos_inputs_new']
                
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '6_cambios_inputs_agregar', dic_cambios = dic_cambios, campos_input_heredados = self.campos_input_heredados)

                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 7:
                               
                str_arquitectura_new = self.str_arquitectura
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.campos_input_heredados = self.campos_input[:]
                try:
                    beta_inputs = self.ffnn.dic_betas[-1]
                except:
                    self.train(n_min)
                    beta_inputs = self.ffnn.dic_betas[-1]
                    
                dic_cambios = agregar_o_eliminar_inputs(self.campos_input, self.all_campos_inputs, 'eliminar', beta_inputs = beta_inputs)
                
                if not dic_cambios['continuar']:
                    continue # Elegir un nuevo x, ya que no hay campos que puedan ser agregados
                
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '7_cambios_inputs_eliminar', dic_cambios = dic_cambios, campos_input_heredados = self.campos_input_heredados)
                self.campos_input = self.ffnn_new.campos_input # Asignacion de campos_inputs nuevos a objeto Busqyeda inteligente              

                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                                
        #print(x, str_arquitectura_new)
        self.str_arquitectura_anterior = self.str_arquitectura
        self.str_arquitectura = str_arquitectura_new # Se hace el cambio
        self.ultimo_cambio = x
        #sys.exit('Revision')
    
    def guardar(self):
        pickle_act(f'{self.cofre}{self.nombre}', variable = self, mode = 'save')
        return None
        
    def ejecutar(self, n_iters, n_min):
        
        entrenar_siguiente = True
        for i in range(n_iters):
            print(f'\n\n\n ######################################################### SIGUIENTE ITERACION {i} ######################################################################## \n\n\n')
            # 0. FFNN actual
            self.ffnn_actual()
            if i % 10 == 0:
                display(self.ffnn.df_info_ffnn.tail())
            # 1. Entrenar por una cantidad definida de epochs
            if entrenar_siguiente:
                self.train(n_min)
            # 1.5 Guardar ffnn actual 
            #\\\ ACTIVAR!!!!\\\
            #print('ACTIVAR!!!!')
            self.ffnn.guardar()
            # 2. Elegir una nueva arquitectura
            if self.inicial: # Si es la primera iteración de todas (solo cuando no existe nada, se guardan los parametros iniciales)
                self.best_test_loss_global = self.best_test_loss_iter # selected
                self.str_arquitectura_best = self.str_arquitectura
                self.best_model = self.ffnn.best_model # Se asigna el nuevo modelo encontrado (con todos sus parámetros) como mejor modelo
                self.mejor_red_neuronal = self.ffnn
                self.inicial = False
                self.elegir_nueva_arquitectura(n_min)
                #sys.exit('Salida para revision')
                continue
            
            # 3. Evaluar el delta en test ecm
            nuevo_delta = False
            if entrenar_siguiente:
                delta = self.best_test_loss_iter - self.best_test_loss_global
                self.dic_movimientos[self.ultimo_cambio] -= delta # cambio en asignacion de dic_movimientos
                nuevo_delta = True
            
            print('\n\n\n')
            print('nuevo_delta', nuevo_delta, 'delta', delta, 'best', self.best_test_loss_global, 'actual', self.best_test_loss_iter)
            
            # Se evalúa si es mejor
            if self.best_test_loss_iter < self.best_test_loss_global:
                print('\n\n Best test error mejora!!!')
                self.best_test_loss_global = self.best_test_loss_iter
                self.str_arquitectura_best = self.str_arquitectura
                self.best_model = self.ffnn.best_model # Se asigna el nuevo modelo encontrado (con todos sus parámetros) como mejor modelo
                self.mejor_red_neuronal = self.ffnn
                print('BASE NUEVA', self.str_arquitectura)
                # Se elige una nueva arquitectura
                self.elegir_nueva_arquitectura(n_min)
                entrenar_siguiente = True
            elif not entrenar_siguiente:
                self.elegir_nueva_arquitectura(n_min)
                entrenar_siguiente = True
            else: # de lo contrario, la base vuelve a ser el caso anterior
                self.str_arquitectura = self.str_arquitectura_anterior 
                print('BASE ANTERIOR', self.str_arquitectura)
                entrenar_siguiente = False
            
            #print('Continuar el 240520. Ver pagina 15')
            #print('Rescatar el error ACTUAL de la red (si existe previamente, de lo contrario, ocupar el error de la última iteracion antes de cambiar la arquitectura)')
            #print('Siempre guardar el best test error global')
            
            print('MEJOR RED NEURONAL HASTA AHORA')
            best_ffnn = self.mejor_red_neuronal
            print(best_ffnn.nombre)
            
            # 4. Guardar busqueda inteligente
            #print('GUARDAR (ACTIVAR)')
            self.guardar()

            print('OK')
        

# Ejecución

## Parámetros de configuración

In [ ]:
metrica = 'Rendimiento'
n_min = 15 # n_min (cuantos epochs seguidos un entrenamiento debe pasar sin mejorar el best ECM para quebrar)
n_iters = 50 # número de iteraciones buscando arquitecturas diferentes
print('aumentar despues, por ahora solo de prueba')

In [ ]:
lista_n_neurs = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96] # input (parametros)

## Ejecución

In [ ]:
if 0 not in lista_n_neurs: # Configuracion de lista_n_neurs: Cuantas neuronas puede tener una capa cualquiera
    lista_n_neurs.append(0)

lista_n_neurs.sort()

In [ ]:
for metrica in ['Rendimiento']:
    busqint = Busqueda_Inteligente('A1', metrica, output_level, cofre, lista_n_neurs)
    busqint.ejecutar(n_iters, n_min)

print('Fin')

# Proceso Predict

In [ ]:
# Elegir el mejor modelo

for metrica in ['Rendimiento']:
    busqint = Busqueda_Inteligente('A1', metrica, output_level, cofre, lista_n_neurs)
    print(busqint.nombre)
    best_ffnn = busqint.mejor_red_neuronal
    print('MEJOR RED NEURONAL ID', best_ffnn.nombre)
    df_x_predict, x_predict = best_ffnn.matrices_predict()
    display(df_x_predict)
    y_predict = best_ffnn.predict_model(x_predict)
    
    df_x_predict['y'] = y_predict.flatten()
    
    display(df_x_predict)
    print('Aqui desnormalizar')
    df_norm = best_ffnn.df_normalizacion.copy()
    df_norm = df_norm[df_norm['CAMPO'] == f'Y_{metrica}_{output_level}']
    y_min, y_max = df_norm['MIN'].values[0], df_norm['MAX'].values[0]
    df_y_predict = df_x_predict[['VALOR', 'DATE', 'y']]
    df_y_predict['y'] = df_y_predict['y'] * (y_max - y_min) + y_min

    max_date = df_y_predict['DATE'].max()
    dia_proyeccion = max_date + dt.timedelta(days = 30)
    df_y_predict = df_y_predict[df_y_predict['DATE'] == max_date].reset_index(drop = True)
    df_y_predict.to_csv(f'{cofre}Inputs_mkw/Rendimiento_{output_level}.csv', sep = ';', decimal = ',', index = False)
    display(df_y_predict)
        
    

In [ ]:
sys.exit()

# Otros (revision etapa predict)

In [ ]:
df_raw_info = pd.DataFrame()
for i in range(len(df_activos)):
    simbolo, nombre = df_activos.loc[i]
    valor = Valor(simbolo, nombre, cofre) # Si no existe el objeto, se crea 
    if len(valor.raw_x) == 0: # No hay datos que aportar
        continue
    #print(simbolo, nombre, len(valor.raw_x))
    new_raw_x = valor.raw_x.copy()
    new_raw_x['VALOR'] = simbolo
    df_raw_info = pd.concat([df_raw_info, new_raw_x], axis = 0)

df_raw_info = df_raw_info[['VALOR', 'DATE', 'NAME', 'X']]
df_raw_info

# Primero, se separan los inputs de los outputs
df_raw_info['NATURALEZA'] = np.where(df_raw_info['NAME'].str[:2] == 'Y_', 'Y', 'X')
df_raw_y = df_raw_info[df_raw_info['NATURALEZA'] == 'Y'].reset_index(drop = True)
df_raw_x = df_raw_info[df_raw_info['NATURALEZA'] == 'X'].reset_index(drop = True)

df_X = df_raw_x.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()
df_Y = df_raw_y.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()

df = df_X.merge(df_Y, on = ['VALOR', 'DATE'], how = 'outer')

while True:
    aprobado = True
    for c in list(set(df.columns) - {'DATE', 'VALOR'}):
        if len(df[df[c].isna()]) > 0:
            aprobado = False
    if aprobado:
        break
    print('A. Espera de completitud de valores en Valor.py: Espera de 30s.')
    time.sleep(30)
            

# Limpieza de df
lista_campos_output = list(set(df_Y.columns) - {'DATE', 'VALOR'})
set_delta_dates = set()
for c in lista_campos_output:

    delta = c.split('_')[-1]
    set_delta_dates.add(delta)
        

print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')  
          
df_predict = df[((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se aislan los datos para después hacer el predict
if len(df_predict) == 0:
    sys.exit('Predict sin datos')
else:
    None
    #print(self.nombre)
    #print(self.df_predict)
df = df[~((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
df_predict